In [3]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torch_directml
import warnings

# Silence the harmless DirectML warning
warnings.filterwarnings("ignore", message=".*aten::lerp.Scalar_out.*")

# ==========================================
# 1. CORE DATA STRUCTURES & MODEL
# ==========================================
class Graph:
    def __init__(self):
        self.num_node = 68  
        self.edges = self._get_edges()
        self.A = self._get_adjacency_matrix()

    def _get_edges(self):
        pose_edges = [(0,1), (1,2), (2,3), (3,7), (0,4), (4,5), (5,6), (6,8), (9,10),
                      (11,12), (11,23), (12,24), (23,24), (11,13), (13,15),
                      (12,14), (14,16), (15,17), (15,19), (15,21),
                      (16,18), (16,20), (16,22)]
        face_edges = [(0, 25)]
        hand_links = [(0,1), (1,2), (2,3), (3,4), (0,5), (5,6), (6,7), (7,8),
                      (5,9), (9,10), (10,11), (11,12), (9,13), (13,14), (14,15), (15,16),
                      (13,17), (0,17), (17,18), (18,19), (19,20)]
        left_hand_edges = [(s + 26, e + 26) for s, e in hand_links]
        right_hand_edges = [(s + 47, e + 47) for s, e in hand_links]
        connection_edges = [(15, 26), (16, 47)]
        return pose_edges + face_edges + left_hand_edges + right_hand_edges + connection_edges

    def _get_adjacency_matrix(self):
        A = np.zeros((self.num_node, self.num_node))
        for i, j in self.edges: A[i, j] = 1; A[j, i] = 1
        return torch.tensor(A, dtype=torch.float32)

class CleanBanglaDataset(Dataset):
    def __init__(self, split_dir, class_to_id):
        self.samples = []
        print(f"📂 Booting RAM-Cache Loader for {os.path.basename(split_dir)}...")
        for class_name in os.listdir(split_dir):
            class_path = os.path.join(split_dir, class_name)
            if not os.path.isdir(class_path): continue
            
            if class_name in class_to_id:
                cid = class_to_id[class_name]
                for f_name in os.listdir(class_path):
                    if f_name.endswith('.npy'):
                        f_path = os.path.join(class_path, f_name)
                        
                        # Load and process ONCE
                        raw_data = np.load(f_path).reshape(90, 68, 3) 
                        raw_data = raw_data - np.mean(raw_data, axis=1, keepdims=True) 
                        raw_data = raw_data.transpose(2, 0, 1)
                        
                        # Save directly to RAM
                        self.samples.append((torch.tensor(raw_data, dtype=torch.float32), torch.tensor(cid, dtype=torch.long)))
                        
        print(f"✅ Successfully cached {len(self.samples)} tensors in Memory!")

    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]

class SpatialGraphConv(nn.Module):
    def __init__(self, in_c, out_c, A):
        super().__init__()
        self.register_buffer('A', A)
        self.conv = nn.Conv2d(in_c, out_c, 1)
    def forward(self, x):
        x = torch.einsum('nctv,vw->nctw', (x, self.A))
        return self.conv(x)

class STGCN_Block(nn.Module):
    def __init__(self, in_c, out_c, A, stride=1, dropout=0.5):
        super().__init__()
        self.sgcn = SpatialGraphConv(in_c, out_c, A)
        self.tgcn = nn.Sequential(
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, (9, 1), (stride, 1), (4, 0)),
            nn.BatchNorm2d(out_c), nn.Dropout(dropout)
        )
        self.res = nn.Sequential(nn.Conv2d(in_c, out_c, 1, (stride, 1)), nn.BatchNorm2d(out_c)) if in_c != out_c or stride != 1 else nn.Identity()
    def forward(self, x): return F.relu(self.tgcn(self.sgcn(x)) + self.res(x))

class BanglaSignSTGCN_Baseline(nn.Module):
    def __init__(self, num_classes, A, dropout_rate=0.5):
        super().__init__()
        self.layer1 = STGCN_Block(3, 64, A, dropout=dropout_rate)
        self.layer2 = STGCN_Block(64, 64, A, dropout=dropout_rate)
        self.layer3 = STGCN_Block(64, 64, A, dropout=dropout_rate)
        self.layer4 = STGCN_Block(64, 128, A, stride=2, dropout=dropout_rate)
        self.layer5 = STGCN_Block(128, 128, A, dropout=dropout_rate)
        self.layer6 = STGCN_Block(128, 128, A, dropout=dropout_rate)
        self.layer7 = STGCN_Block(128, 256, A, stride=2, dropout=dropout_rate)
        self.layer8 = STGCN_Block(256, 256, A, dropout=dropout_rate)
        self.layer9 = STGCN_Block(256, 256, A, dropout=dropout_rate)
        self.fcn = nn.Conv2d(256, num_classes, 1)
        
    def forward(self, x):
        for l in [self.layer1,self.layer2,self.layer3,self.layer4,self.layer5,self.layer6,self.layer7,self.layer8,self.layer9]: 
            x = l(x)
        x = F.avg_pool2d(x, x.size()[2:])
        return self.fcn(x).view(x.size(0), -1)

def train_baseline_model():
    # 🟢 REPRODUCIBILITY LOCK
    seed = 42
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    # SET YOUR DIRECTORY HERE
    DATASET_DIR = r"C:\Users\User\Documents\Personal Akams\Thesis\Final_Thesis_Dataset_1\Fold_1"
    TRAIN_DIR = os.path.join(DATASET_DIR, "train")
    VAL_DIR = os.path.join(DATASET_DIR, "val")
    MODELS_DIR = "1.Absolute_Baseline_Models"
    os.makedirs(MODELS_DIR, exist_ok=True)
    
    EPOCHS = 50
    BATCH_SIZE = 32
    LEARNING_RATE = 0.001
    DROPOUT = 0.5
    WEIGHT_DECAY = 1e-4
    
    dml = torch_directml.device()
    print(f"\n🚀 STRICT BASELINE ENGINE STARTED. Using GPU: {torch_directml.device_name(0)}")
    print(f"🔒 Random Seed locked to {seed} for strict reproducibility.")

    all_classes = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
    class_to_id = {cls_name: idx for idx, cls_name in enumerate(all_classes)}
    id_to_class = {idx: cls_name for cls_name, idx in class_to_id.items()} 
    num_classes = len(all_classes)
    
    homophone_pairs = [("0_Lefthand", "O_Lefthand"), ("0_Righthand", "O_Righthand"),
                       ("2_Lefthand", "V_Lefthand"), ("2_Righthand", "V_Righthand")]
    
    allowed_confusions = []
    for a, b in homophone_pairs:
        if a in class_to_id and b in class_to_id:
            allowed_confusions.append(set([class_to_id[a], class_to_id[b]]))

    base_allowed_confusions = []
    for a, b in homophone_pairs:
        base_a = a.split('_')[0] if '_' in a else a
        base_b = b.split('_')[0] if '_' in b else b
        pair = set([base_a, base_b])
        if pair not in base_allowed_confusions:
            base_allowed_confusions.append(pair)

    train_data = CleanBanglaDataset(TRAIN_DIR, class_to_id)
    val_data = CleanBanglaDataset(VAL_DIR, class_to_id)

    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    graph = Graph()
    model = BanglaSignSTGCN_Baseline(num_classes, graph.A, dropout_rate=DROPOUT).to(dml)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0

    for epoch in range(EPOCHS):
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        
        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(dml), labels.to(dml)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * inputs.size(0)
            _, pred = outputs.max(1)
            train_total += labels.size(0)
            train_correct += pred.eq(labels).sum().item()

        epoch_train_acc = 100. * train_correct / train_total

        model.eval()
        val_loss, strict_correct, relaxed_correct, base_sign_correct, val_total = 0.0, 0, 0, 0, 0
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(dml), labels.to(dml)
                outputs = model(inputs)
                
                _, top1_preds = outputs.max(1) 
                val_total += labels.size(0)
                
                for i in range(labels.size(0)):
                    true_id = labels[i].item()
                    top1_id = top1_preds[i].item()
                    
                    true_name = id_to_class[true_id]
                    pred_name = id_to_class[top1_id]
                    
                    # STRICT & RELAXED LOGIC
                    if top1_id == true_id:
                        strict_correct += 1
                        relaxed_correct += 1
                    else:
                        prediction_pair = set([true_id, top1_id])
                        if prediction_pair in allowed_confusions:
                            relaxed_correct += 1
                            
                    # BASE SIGN LOGIC
                    true_base = true_name.split('_')[0] if '_' in true_name else true_name
                    pred_base = pred_name.split('_')[0] if '_' in pred_name else pred_name
                    
                    if true_base == pred_base or set([true_base, pred_base]) in base_allowed_confusions:
                        base_sign_correct += 1
                
        strict_acc = 100. * strict_correct / val_total
        relaxed_acc = 100. * relaxed_correct / val_total
        base_sign_acc = 100. * base_sign_correct / val_total 

        print(f"Epoch [{epoch+1:02d}/{EPOCHS}] | Train Acc: {epoch_train_acc:.2f}% | "
              f"Val Strict: {strict_acc:.2f}% | Relaxed: {relaxed_acc:.2f}% -> Base Sign: {base_sign_acc:.2f}%")

        if relaxed_acc > best_val_acc:
            best_val_acc = relaxed_acc
            save_path = os.path.join(MODELS_DIR, "best_baseline_stgcn.pth")
            torch.save(model.state_dict(), save_path)
            
        scheduler.step()

if __name__ == "__main__":
    train_baseline_model()


🚀 STRICT BASELINE ENGINE STARTED. Using GPU: AMD Radeon RX 7900 GRE 
🔒 Random Seed locked to 42 for strict reproducibility.
📂 Booting RAM-Cache Loader for train...
✅ Successfully cached 34200 tensors in Memory!
📂 Booting RAM-Cache Loader for val...
✅ Successfully cached 8681 tensors in Memory!
Epoch [01/50] | Train Acc: 8.51% | Val Strict: 19.24% | Relaxed: 19.24% -> Base Sign: 21.32%
Epoch [02/50] | Train Acc: 32.06% | Val Strict: 38.66% | Relaxed: 38.75% -> Base Sign: 40.50%
Epoch [03/50] | Train Acc: 48.08% | Val Strict: 51.60% | Relaxed: 51.61% -> Base Sign: 52.99%
Epoch [04/50] | Train Acc: 58.78% | Val Strict: 54.96% | Relaxed: 55.13% -> Base Sign: 56.28%
Epoch [05/50] | Train Acc: 66.74% | Val Strict: 57.49% | Relaxed: 57.60% -> Base Sign: 58.76%
Epoch [06/50] | Train Acc: 72.77% | Val Strict: 62.70% | Relaxed: 63.05% -> Base Sign: 63.90%
Epoch [07/50] | Train Acc: 76.58% | Val Strict: 71.21% | Relaxed: 71.36% -> Base Sign: 72.12%
Epoch [08/50] | Train Acc: 79.90% | Val Strict:

Baseline model Graph generation ()

In [4]:
import matplotlib.pyplot as plt
import re
import os

# The exact console output you provided
log_data = """
Epoch [01/50] | Train Acc: 8.51% | Val Strict: 19.24% | Relaxed: 19.24% -> Base Sign: 21.32%
Epoch [02/50] | Train Acc: 32.06% | Val Strict: 38.66% | Relaxed: 38.75% -> Base Sign: 40.50%
Epoch [03/50] | Train Acc: 48.08% | Val Strict: 51.60% | Relaxed: 51.61% -> Base Sign: 52.99%
Epoch [04/50] | Train Acc: 58.78% | Val Strict: 54.96% | Relaxed: 55.13% -> Base Sign: 56.28%
Epoch [05/50] | Train Acc: 66.74% | Val Strict: 57.49% | Relaxed: 57.60% -> Base Sign: 58.76%
Epoch [06/50] | Train Acc: 72.77% | Val Strict: 62.70% | Relaxed: 63.05% -> Base Sign: 63.90%
Epoch [07/50] | Train Acc: 76.58% | Val Strict: 71.21% | Relaxed: 71.36% -> Base Sign: 72.12%
Epoch [08/50] | Train Acc: 79.90% | Val Strict: 69.54% | Relaxed: 69.82% -> Base Sign: 71.04%
Epoch [09/50] | Train Acc: 82.44% | Val Strict: 73.25% | Relaxed: 73.63% -> Base Sign: 74.37%
Epoch [10/50] | Train Acc: 84.82% | Val Strict: 74.75% | Relaxed: 75.16% -> Base Sign: 75.94%
Epoch [11/50] | Train Acc: 86.11% | Val Strict: 64.73% | Relaxed: 64.83% -> Base Sign: 65.48%
Epoch [12/50] | Train Acc: 87.57% | Val Strict: 70.88% | Relaxed: 71.36% -> Base Sign: 72.15%
Epoch [13/50] | Train Acc: 88.74% | Val Strict: 76.07% | Relaxed: 76.64% -> Base Sign: 77.33%
Epoch [14/50] | Train Acc: 89.56% | Val Strict: 76.60% | Relaxed: 76.83% -> Base Sign: 77.58%
Epoch [15/50] | Train Acc: 90.47% | Val Strict: 67.81% | Relaxed: 67.85% -> Base Sign: 68.52%
Epoch [16/50] | Train Acc: 95.23% | Val Strict: 77.38% | Relaxed: 77.63% -> Base Sign: 78.22%
Epoch [17/50] | Train Acc: 95.31% | Val Strict: 83.10% | Relaxed: 83.50% -> Base Sign: 83.86%
Epoch [18/50] | Train Acc: 95.26% | Val Strict: 86.84% | Relaxed: 87.27% -> Base Sign: 87.66%
Epoch [19/50] | Train Acc: 95.39% | Val Strict: 86.90% | Relaxed: 87.39% -> Base Sign: 88.12%
Epoch [20/50] | Train Acc: 95.64% | Val Strict: 79.40% | Relaxed: 79.53% -> Base Sign: 79.86%
Epoch [21/50] | Train Acc: 95.76% | Val Strict: 78.75% | Relaxed: 78.98% -> Base Sign: 79.52%
Epoch [22/50] | Train Acc: 95.78% | Val Strict: 82.64% | Relaxed: 82.94% -> Base Sign: 83.45%
Epoch [23/50] | Train Acc: 95.94% | Val Strict: 81.27% | Relaxed: 81.67% -> Base Sign: 82.21%
Epoch [24/50] | Train Acc: 95.97% | Val Strict: 81.36% | Relaxed: 81.74% -> Base Sign: 82.17%
Epoch [25/50] | Train Acc: 96.47% | Val Strict: 78.55% | Relaxed: 78.85% -> Base Sign: 79.44%
Epoch [26/50] | Train Acc: 95.91% | Val Strict: 81.95% | Relaxed: 82.19% -> Base Sign: 82.53%
Epoch [27/50] | Train Acc: 96.47% | Val Strict: 87.81% | Relaxed: 88.18% -> Base Sign: 88.68%
Epoch [28/50] | Train Acc: 96.49% | Val Strict: 88.48% | Relaxed: 88.94% -> Base Sign: 89.31%
Epoch [29/50] | Train Acc: 96.72% | Val Strict: 84.40% | Relaxed: 84.78% -> Base Sign: 85.16%
Epoch [30/50] | Train Acc: 96.62% | Val Strict: 83.45% | Relaxed: 83.73% -> Base Sign: 84.22%
Epoch [31/50] | Train Acc: 98.49% | Val Strict: 88.48% | Relaxed: 88.85% -> Base Sign: 89.40%
Epoch [32/50] | Train Acc: 98.57% | Val Strict: 86.68% | Relaxed: 87.16% -> Base Sign: 87.51%
Epoch [33/50] | Train Acc: 98.42% | Val Strict: 88.08% | Relaxed: 88.50% -> Base Sign: 88.84%
Epoch [34/50] | Train Acc: 98.42% | Val Strict: 90.38% | Relaxed: 90.59% -> Base Sign: 91.01%
Epoch [35/50] | Train Acc: 98.63% | Val Strict: 85.75% | Relaxed: 86.14% -> Base Sign: 86.49%
Epoch [36/50] | Train Acc: 98.55% | Val Strict: 87.58% | Relaxed: 87.79% -> Base Sign: 88.31%
Epoch [37/50] | Train Acc: 98.40% | Val Strict: 83.93% | Relaxed: 84.26% -> Base Sign: 84.81%
Epoch [38/50] | Train Acc: 98.52% | Val Strict: 87.74% | Relaxed: 88.03% -> Base Sign: 88.39%
Epoch [39/50] | Train Acc: 98.68% | Val Strict: 91.95% | Relaxed: 92.34% -> Base Sign: 92.52%
Epoch [40/50] | Train Acc: 98.55% | Val Strict: 89.15% | Relaxed: 89.59% -> Base Sign: 89.79%
Epoch [41/50] | Train Acc: 98.57% | Val Strict: 90.04% | Relaxed: 90.35% -> Base Sign: 90.67%
Epoch [42/50] | Train Acc: 98.70% | Val Strict: 89.08% | Relaxed: 89.32% -> Base Sign: 89.62%
Epoch [43/50] | Train Acc: 98.59% | Val Strict: 86.29% | Relaxed: 86.67% -> Base Sign: 86.94%
Epoch [44/50] | Train Acc: 98.73% | Val Strict: 91.30% | Relaxed: 91.60% -> Base Sign: 91.87%
Epoch [45/50] | Train Acc: 98.61% | Val Strict: 90.92% | Relaxed: 91.23% -> Base Sign: 91.56%
Epoch [46/50] | Train Acc: 99.49% | Val Strict: 92.19% | Relaxed: 92.42% -> Base Sign: 92.78%
Epoch [47/50] | Train Acc: 99.44% | Val Strict: 92.90% | Relaxed: 93.26% -> Base Sign: 93.57%
Epoch [48/50] | Train Acc: 99.45% | Val Strict: 92.52% | Relaxed: 92.77% -> Base Sign: 93.15%
Epoch [49/50] | Train Acc: 99.49% | Val Strict: 92.36% | Relaxed: 92.66% -> Base Sign: 92.90%
Epoch [50/50] | Train Acc: 99.56% | Val Strict: 92.19% | Relaxed: 92.50% -> Base Sign: 92.75%
"""

history_train = []
history_strict = []
history_relaxed = []
history_base = []

# Regex to parse the numbers from the log lines
pattern = r"Train Acc: ([\d.]+)% \| Val Strict: ([\d.]+)% \| Relaxed: ([\d.]+)% -> Base Sign: ([\d.]+)%"

for line in log_data.strip().split('\n'):
    match = re.search(pattern, line)
    if match:
        history_train.append(float(match.group(1)))
        history_strict.append(float(match.group(2)))
        history_relaxed.append(float(match.group(3)))
        history_base.append(float(match.group(4)))

EPOCHS = len(history_train)
MODELS_DIR = "1.Absolute_Baseline_Models"
os.makedirs(MODELS_DIR, exist_ok=True)

print("⏳ Generating Detailed Accuracy Graph...")
# 1. Detailed Graph
plt.figure(figsize=(10, 6))
plt.plot(range(1, EPOCHS + 1), history_train, label='Train Accuracy', color='blue')
plt.plot(range(1, EPOCHS + 1), history_strict, label='Val Strict', color='red')
plt.plot(range(1, EPOCHS + 1), history_relaxed, label='Val Relaxed', color='green')
plt.plot(range(1, EPOCHS + 1), history_base, label='Val Base Sign', color='orange', linestyle='--')
plt.title('Strict Baseline Accuracy (Detailed Metrics)')
plt.xlabel('Epochs')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True)
plt.savefig(os.path.join(MODELS_DIR, "accuracy_graph_baseline_detailed.png"))
plt.close()

print("⏳ Generating Traditional Accuracy Graph...")
# 2. Strict Train vs Val Graph (Presentation Ready)
plt.figure(figsize=(10, 6), dpi=300)
plt.plot(range(1, EPOCHS + 1), history_train, label='Training Accuracy', color='#1f77b4', linewidth=2.5)
plt.plot(range(1, EPOCHS + 1), history_strict, label='Validation Accuracy (Strict)', color='#d62728', linewidth=2.5)
plt.title('Train vs Strict Validation (Absolute Baseline)', fontsize=14, fontweight='bold')
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Accuracy (%)', fontsize=12)
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, "accuracy_graph_baseline_traditional.png"))
plt.close()

print("✅ Graphs saved successfully in 1.Absolute_Baseline_Models directory!")

⏳ Generating Detailed Accuracy Graph...
⏳ Generating Traditional Accuracy Graph...
✅ Graphs saved successfully in 1.Absolute_Baseline_Models directory!


Experment 2 : introducing cosine annealing , optimizer adamw and smooth labelling

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torch_directml
import warnings
import matplotlib.pyplot as plt

# 🟢 Silence the harmless DirectML warning
warnings.filterwarnings("ignore", message=".*aten::lerp.Scalar_out.*")

# ==========================================
# 1. CORE DATA STRUCTURES & MODEL
# ==========================================
class Graph:
    def __init__(self):
        self.num_node = 68  
        self.edges = self._get_edges()
        self.A = self._get_adjacency_matrix()

    def _get_edges(self):
        pose_edges = [
            (0,1), (1,2), (2,3), (3,7),
            (0,4), (4,5), (5,6), (6,8),
            (9,10),
            (11,12), (11,23), (12,24), (23,24),
            (11,13), (13,15),
            (12,14), (14,16),
            (15,17), (15,19), (15,21),
            (16,18), (16,20), (16,22)
        ]
        face_edges = [(0, 25)]
        hand_links = [(0,1), (1,2), (2,3), (3,4),
                      (0,5), (5,6), (6,7), (7,8),
                      (5,9), (9,10), (10,11), (11,12),
                      (9,13), (13,14), (14,15), (15,16),
                      (13,17), (0,17), (17,18), (18,19), (19,20)]
        left_hand_edges = [(s + 26, e + 26) for s, e in hand_links]
        right_hand_edges = [(s + 47, e + 47) for s, e in hand_links]
        connection_edges = [(15, 26), (16, 47)]
        
        return pose_edges + face_edges + left_hand_edges + right_hand_edges + connection_edges

    def _get_adjacency_matrix(self):
        A = np.zeros((self.num_node, self.num_node))
        for i, j in self.edges:
            A[i, j] = 1; A[j, i] = 1
        return torch.tensor(A, dtype=torch.float32)

class CleanBanglaDataset(Dataset):
    def __init__(self, split_dir, class_to_id):
        self.samples = []
        
        print(f"📂 Booting RAM-Cache Loader for {os.path.basename(split_dir)}...")
        print(f"⏳ Loading arrays into RAM (This takes ~1 min but makes epochs 10x faster)...")
        
        for class_name in os.listdir(split_dir):
            class_path = os.path.join(split_dir, class_name)
            if not os.path.isdir(class_path): continue
            
            if class_name in class_to_id:
                cid = class_to_id[class_name]
                for f_name in os.listdir(class_path):
                    if f_name.endswith('.npy'):
                        f_path = os.path.join(class_path, f_name)
                        
                        raw_data = np.load(f_path).reshape(90, 68, 3) 
                        raw_data = raw_data - np.mean(raw_data, axis=1, keepdims=True) 
                        raw_data = raw_data.transpose(2, 0, 1)
                        self.samples.append((raw_data, cid))
                        
        print(f"✅ Successfully cached {len(self.samples)} tensors in Memory!")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        data_cached, label = self.samples[idx]
        data = data_cached.copy()
        return torch.tensor(data, dtype=torch.float32), torch.tensor(label, dtype=torch.long)

class SpatialGraphConv(nn.Module):
    def __init__(self, in_c, out_c, A):
        super().__init__()
        self.register_buffer('A', A)
        self.conv = nn.Conv2d(in_c, out_c, 1)
    def forward(self, x):
        x = torch.einsum('nctv,vw->nctw', (x, self.A))
        return self.conv(x)

class STGCN_Block(nn.Module):
    def __init__(self, in_c, out_c, A, stride=1, dropout=0.4):
        super().__init__()
        self.sgcn = SpatialGraphConv(in_c, out_c, A)
        self.tgcn = nn.Sequential(
            nn.BatchNorm2d(out_c), 
            nn.ReLU(inplace=True), 
            # 🟢 REVERTED: Back to Baseline 9-Frame Temporal Window
            nn.Conv2d(out_c, out_c, (9, 1), (stride, 1), (4, 0)),
            nn.BatchNorm2d(out_c), 
            nn.Dropout(dropout)
        )
        self.res = nn.Sequential(nn.Conv2d(in_c, out_c, 1, (stride, 1)), nn.BatchNorm2d(out_c)) if in_c != out_c or stride != 1 else nn.Identity()
        
    def forward(self, x): 
        return F.relu(self.tgcn(self.sgcn(x)) + self.res(x)) 

class BanglaSignSTGCN(nn.Module):
    def __init__(self, num_classes, A, dropout_rate=0.4):
        super().__init__()
        self.layer1 = STGCN_Block(3, 64, A, dropout=dropout_rate)
        self.layer2 = STGCN_Block(64, 64, A, dropout=dropout_rate)
        self.layer3 = STGCN_Block(64, 64, A, dropout=dropout_rate)
        self.layer4 = STGCN_Block(64, 128, A, stride=2, dropout=dropout_rate)
        self.layer5 = STGCN_Block(128, 128, A, dropout=dropout_rate)
        self.layer6 = STGCN_Block(128, 128, A, dropout=dropout_rate)
        self.layer7 = STGCN_Block(128, 256, A, stride=2, dropout=dropout_rate)
        self.layer8 = STGCN_Block(256, 256, A, dropout=dropout_rate)
        self.layer9 = STGCN_Block(256, 256, A, dropout=dropout_rate)
        self.fcn = nn.Conv2d(256, num_classes, 1)
    def forward(self, x):
        for l in [self.layer1,self.layer2,self.layer3,self.layer4,self.layer5,self.layer6,self.layer7,self.layer8,self.layer9]: x = l(x)
        x = F.avg_pool2d(x, x.size()[2:])
        return self.fcn(x).view(x.size(0), -1)

# ==========================================
# 2. STEPPING STONE TRAINING ENGINE
# ==========================================
def train_baseline_model():
    # 🟢 REPRODUCIBILITY LOCK
    seed = 42
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    DATASET_DIR = r"C:\Users\User\Documents\Personal Akams\Thesis\Final_Thesis_Dataset_1\Fold_1"
    TRAIN_DIR = os.path.join(DATASET_DIR, "train")
    VAL_DIR = os.path.join(DATASET_DIR, "val")
    
    MODELS_DIR = "Stepping_Stone_Models"
    os.makedirs(MODELS_DIR, exist_ok=True)
    
    # 🟢 EXPERT ENGINE + BASELINE SETTINGS
    EPOCHS = 50
    BATCH_SIZE = 32
    LEARNING_RATE = 0.001 # 🟢 REVERTED: Back to Baseline LR
    DROPOUT = 0.4
    WEIGHT_DECAY = 1e-4
    NOISE_LEVEL = 0.005 
    
    dml = torch_directml.device()
    print(f"\n🚀 STEPPING STONE ENGINE STARTED. Using GPU: {torch_directml.device_name(0)}")
    print(f"🔒 Random Seed locked to {seed} for strict reproducibility.")

    all_classes = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
    class_to_id = {cls_name: idx for idx, cls_name in enumerate(all_classes)}
    id_to_class = {idx: cls_name for cls_name, idx in class_to_id.items()} 
    num_classes = len(all_classes)
    
    homophone_pairs = [
        ("0_Lefthand", "O_Lefthand"),
        ("0_Righthand", "O_Righthand"),
        ("2_Lefthand", "V_Lefthand"),
        ("2_Righthand", "V_Righthand")
    ]
    
    allowed_confusions = []
    for a, b in homophone_pairs:
        if a in class_to_id and b in class_to_id:
            allowed_confusions.append(set([class_to_id[a], class_to_id[b]]))

    base_allowed_confusions = []
    for a, b in homophone_pairs:
        base_a = a.split('_')[0] if '_' in a else a
        base_b = b.split('_')[0] if '_' in b else b
        pair = set([base_a, base_b])
        if pair not in base_allowed_confusions:
            base_allowed_confusions.append(pair)

    train_data = CleanBanglaDataset(TRAIN_DIR, class_to_id)
    val_data = CleanBanglaDataset(VAL_DIR, class_to_id)

    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    graph = Graph()
    model = BanglaSignSTGCN(num_classes, graph.A, dropout_rate=DROPOUT).to(dml)
    
    # 🟢 EXPERT ENGINE UPGRADES
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, foreach=False)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1) # The third "special change"

    best_val_acc = 0.0

    history_train = []
    history_strict = []
    history_relaxed = []
    history_base = []

    for epoch in range(EPOCHS):
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        
        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs = inputs.to(dml, non_blocking=True)
            labels = labels.to(dml, non_blocking=True)
            
            if NOISE_LEVEL > 0.0:
                noise = torch.randn(*inputs.shape, device=dml) * NOISE_LEVEL
                inputs = inputs + noise
                
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * inputs.size(0)
            _, pred = outputs.max(1)
            train_total += labels.size(0)
            train_correct += pred.eq(labels).sum().item()

        epoch_train_acc = 100. * train_correct / train_total

        model.eval()
        
        val_loss, strict_correct, relaxed_correct, base_sign_correct, val_total = 0.0, 0, 0, 0, 0
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(dml, non_blocking=True)
                labels = labels.to(dml, non_blocking=True)
                
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                val_total += labels.size(0)
                
                _, top1_preds = outputs.max(1) 
                
                for i in range(labels.size(0)):
                    true_id = labels[i].item()
                    top1_id = top1_preds[i].item()
                    
                    true_name = id_to_class[true_id]
                    pred_name = id_to_class[top1_id]
                    
                    if top1_id == true_id:
                        strict_correct += 1
                        relaxed_correct += 1
                    else:
                        prediction_pair = set([true_id, top1_id])
                        if prediction_pair in allowed_confusions:
                            relaxed_correct += 1
                            
                    true_base = true_name.split('_')[0] if '_' in true_name else true_name
                    pred_base = pred_name.split('_')[0] if '_' in pred_name else pred_name
                    
                    if true_base == pred_base or set([true_base, pred_base]) in base_allowed_confusions:
                        base_sign_correct += 1
                
        strict_acc = 100. * strict_correct / val_total
        relaxed_acc = 100. * relaxed_correct / val_total
        base_sign_acc = 100. * base_sign_correct / val_total 

        history_train.append(epoch_train_acc)
        history_strict.append(strict_acc)
        history_relaxed.append(relaxed_acc)
        history_base.append(base_sign_acc)

        print(f"Epoch [{epoch+1:02d}/{EPOCHS}] | Train: {epoch_train_acc:.2f}% | "
              f"Val Strict: {strict_acc:.2f}% | Relaxed: {relaxed_acc:.2f}% -> Base Sign: {base_sign_acc:.2f}%")

        if relaxed_acc > best_val_acc: 
            best_val_acc = relaxed_acc
            save_path = os.path.join(MODELS_DIR, "stepping_stone_relu_9frame.pth")
            torch.save(model.state_dict(), save_path)
            
        scheduler.step()

    print("\n" + "="*60)
    print(f"🎉 Training Complete! Best Relaxed Val Accuracy: {best_val_acc:.2f}%")
    print("="*60)

    print("⏳ Generating Detailed Accuracy Graph...")
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, EPOCHS + 1), history_train, label='Train Accuracy', color='blue')
    plt.plot(range(1, EPOCHS + 1), history_strict, label='Simple Val Acc', color='red')
    plt.plot(range(1, EPOCHS + 1), history_relaxed, label='Val Relaxed', color='green')
    plt.plot(range(1, EPOCHS + 1), history_base, label='Val Base Sign', color='orange', linestyle='--')
    
    plt.title('Stepping Stone Model Accuracy (Detailed Metrics)')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True)
    
    graph_path_detailed = os.path.join(MODELS_DIR, "accuracy_graph_stepping_stone_detailed.png")
    plt.savefig(graph_path_detailed)
    plt.close()

if __name__ == "__main__":
    train_baseline_model()


🚀 STEPPING STONE ENGINE STARTED. Using GPU: AMD Radeon RX 7900 GRE 
🔒 Random Seed locked to 42 for strict reproducibility.
📂 Booting RAM-Cache Loader for train...
⏳ Loading arrays into RAM (This takes ~1 min but makes epochs 10x faster)...
✅ Successfully cached 34200 tensors in Memory!
📂 Booting RAM-Cache Loader for val...
⏳ Loading arrays into RAM (This takes ~1 min but makes epochs 10x faster)...
✅ Successfully cached 8681 tensors in Memory!
Epoch [01/50] | Train: 9.53% | Val Strict: 24.82% | Relaxed: 24.90% -> Base Sign: 27.16%
Epoch [02/50] | Train: 37.02% | Val Strict: 47.09% | Relaxed: 47.39% -> Base Sign: 48.83%
Epoch [03/50] | Train: 57.15% | Val Strict: 61.76% | Relaxed: 61.97% -> Base Sign: 63.40%
Epoch [04/50] | Train: 69.44% | Val Strict: 63.32% | Relaxed: 63.40% -> Base Sign: 64.84%
Epoch [05/50] | Train: 77.40% | Val Strict: 74.62% | Relaxed: 74.90% -> Base Sign: 75.92%
Epoch [06/50] | Train: 82.96% | Val Strict: 77.03% | Relaxed: 77.53% -> Base Sign: 78.11%
Epoch [07/50

Swish function experiment

In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torch_directml
import warnings
import matplotlib.pyplot as plt

# 🟢 Silence the harmless DirectML warning
warnings.filterwarnings("ignore", message=".*aten::lerp.Scalar_out.*")

# ==========================================
# 1. CORE DATA STRUCTURES & MODEL
# ==========================================
class Graph:
    def __init__(self):
        self.num_node = 68  
        self.edges = self._get_edges()
        self.A = self._get_adjacency_matrix()

    def _get_edges(self):
        pose_edges = [
            (0,1), (1,2), (2,3), (3,7),
            (0,4), (4,5), (5,6), (6,8),
            (9,10),
            (11,12), (11,23), (12,24), (23,24),
            (11,13), (13,15),
            (12,14), (14,16),
            (15,17), (15,19), (15,21),
            (16,18), (16,20), (16,22)
        ]
        face_edges = [(0, 25)]
        hand_links = [(0,1), (1,2), (2,3), (3,4),
                      (0,5), (5,6), (6,7), (7,8),
                      (5,9), (9,10), (10,11), (11,12),
                      (9,13), (13,14), (14,15), (15,16),
                      (13,17), (0,17), (17,18), (18,19), (19,20)]
        left_hand_edges = [(s + 26, e + 26) for s, e in hand_links]
        right_hand_edges = [(s + 47, e + 47) for s, e in hand_links]
        connection_edges = [(15, 26), (16, 47)]
        
        return pose_edges + face_edges + left_hand_edges + right_hand_edges + connection_edges

    def _get_adjacency_matrix(self):
        A = np.zeros((self.num_node, self.num_node))
        for i, j in self.edges:
            A[i, j] = 1; A[j, i] = 1
        return torch.tensor(A, dtype=torch.float32)

class CleanBanglaDataset(Dataset):
    def __init__(self, split_dir, class_to_id):
        self.samples = []
        
        print(f"📂 Booting RAM-Cache Loader for {os.path.basename(split_dir)}...")
        print(f"⏳ Loading arrays into RAM (This takes ~1 min but makes epochs 10x faster)...")
        
        for class_name in os.listdir(split_dir):
            class_path = os.path.join(split_dir, class_name)
            if not os.path.isdir(class_path): continue
            
            if class_name in class_to_id:
                cid = class_to_id[class_name]
                for f_name in os.listdir(class_path):
                    if f_name.endswith('.npy'):
                        f_path = os.path.join(class_path, f_name)
                        
                        raw_data = np.load(f_path).reshape(90, 68, 3) 
                        raw_data = raw_data - np.mean(raw_data, axis=1, keepdims=True) 
                        raw_data = raw_data.transpose(2, 0, 1)
                        self.samples.append((raw_data, cid))
                        
        print(f"✅ Successfully cached {len(self.samples)} tensors in Memory!")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        data_cached, label = self.samples[idx]
        data = data_cached.copy()
        return torch.tensor(data, dtype=torch.float32), torch.tensor(label, dtype=torch.long)

class SpatialGraphConv(nn.Module):
    def __init__(self, in_c, out_c, A):
        super().__init__()
        self.register_buffer('A', A)
        self.conv = nn.Conv2d(in_c, out_c, 1)
    def forward(self, x):
        x = torch.einsum('nctv,vw->nctw', (x, self.A))
        return self.conv(x)

class STGCN_Block(nn.Module):
    def __init__(self, in_c, out_c, A, stride=1, dropout=0.4):
        super().__init__()
        self.sgcn = SpatialGraphConv(in_c, out_c, A)
        self.tgcn = nn.Sequential(
            nn.BatchNorm2d(out_c), 
            nn.SiLU(inplace=True), # 🟢 CHANGED: Swish (SiLU) Activation
            nn.Conv2d(out_c, out_c, (9, 1), (stride, 1), (4, 0)),
            nn.BatchNorm2d(out_c), 
            nn.Dropout(dropout)
        )
        self.res = nn.Sequential(nn.Conv2d(in_c, out_c, 1, (stride, 1)), nn.BatchNorm2d(out_c)) if in_c != out_c or stride != 1 else nn.Identity()
        
    def forward(self, x): 
        # 🟢 CHANGED: Swish (SiLU) applied to residual output
        return F.silu(self.tgcn(self.sgcn(x)) + self.res(x)) 

class BanglaSignSTGCN(nn.Module):
    def __init__(self, num_classes, A, dropout_rate=0.4):
        super().__init__()
        self.layer1 = STGCN_Block(3, 64, A, dropout=dropout_rate)
        self.layer2 = STGCN_Block(64, 64, A, dropout=dropout_rate)
        self.layer3 = STGCN_Block(64, 64, A, dropout=dropout_rate)
        self.layer4 = STGCN_Block(64, 128, A, stride=2, dropout=dropout_rate)
        self.layer5 = STGCN_Block(128, 128, A, dropout=dropout_rate)
        self.layer6 = STGCN_Block(128, 128, A, dropout=dropout_rate)
        self.layer7 = STGCN_Block(128, 256, A, stride=2, dropout=dropout_rate)
        self.layer8 = STGCN_Block(256, 256, A, dropout=dropout_rate)
        self.layer9 = STGCN_Block(256, 256, A, dropout=dropout_rate)
        self.fcn = nn.Conv2d(256, num_classes, 1)
    def forward(self, x):
        for l in [self.layer1,self.layer2,self.layer3,self.layer4,self.layer5,self.layer6,self.layer7,self.layer8,self.layer9]: x = l(x)
        x = F.avg_pool2d(x, x.size()[2:])
        return self.fcn(x).view(x.size(0), -1)

# ==========================================
# 2. EXPERT TRAINING ENGINE (SWISH ABLATION)
# ==========================================
def train_baseline_model():
    # 🟢 REPRODUCIBILITY LOCK
    seed = 42
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    DATASET_DIR = r"C:\Users\User\Documents\Personal Akams\Thesis\Final_Thesis_Dataset_1\Fold_1"
    TRAIN_DIR = os.path.join(DATASET_DIR, "train")
    VAL_DIR = os.path.join(DATASET_DIR, "val")
    
    MODELS_DIR = "Swish Function Experiment"
    os.makedirs(MODELS_DIR, exist_ok=True)
    
    # 🟢 EXPERT ENGINE SETTINGS
    EPOCHS = 50
    BATCH_SIZE = 32
    LEARNING_RATE = 0.001 
    DROPOUT = 0.4
    WEIGHT_DECAY = 1e-4
    NOISE_LEVEL = 0.005 
    
    dml = torch_directml.device()
    print(f"\n🚀 SWISH ACTIVATION STUDY STARTED. Using GPU: {torch_directml.device_name(0)}")
    print(f"🔒 Random Seed locked to {seed} for strict reproducibility.")

    all_classes = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
    class_to_id = {cls_name: idx for idx, cls_name in enumerate(all_classes)}
    id_to_class = {idx: cls_name for cls_name, idx in class_to_id.items()} 
    num_classes = len(all_classes)
    
    homophone_pairs = [
        ("0_Lefthand", "O_Lefthand"),
        ("0_Righthand", "O_Righthand"),
        ("2_Lefthand", "V_Lefthand"),
        ("2_Righthand", "V_Righthand")
    ]
    
    allowed_confusions = []
    for a, b in homophone_pairs:
        if a in class_to_id and b in class_to_id:
            allowed_confusions.append(set([class_to_id[a], class_to_id[b]]))

    base_allowed_confusions = []
    for a, b in homophone_pairs:
        base_a = a.split('_')[0] if '_' in a else a
        base_b = b.split('_')[0] if '_' in b else b
        pair = set([base_a, base_b])
        if pair not in base_allowed_confusions:
            base_allowed_confusions.append(pair)

    train_data = CleanBanglaDataset(TRAIN_DIR, class_to_id)
    val_data = CleanBanglaDataset(VAL_DIR, class_to_id)

    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    graph = Graph()
    model = BanglaSignSTGCN(num_classes, graph.A, dropout_rate=DROPOUT).to(dml)
    
    # 🟢 EXPERT ENGINE UPGRADES
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, foreach=False)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    best_val_acc = 0.0

    history_train = []
    history_strict = []
    history_relaxed = []
    history_base = []

    for epoch in range(EPOCHS):
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        
        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs = inputs.to(dml, non_blocking=True)
            labels = labels.to(dml, non_blocking=True)
            
            if NOISE_LEVEL > 0.0:
                noise = torch.randn(*inputs.shape, device=dml) * NOISE_LEVEL
                inputs = inputs + noise
                
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * inputs.size(0)
            _, pred = outputs.max(1)
            train_total += labels.size(0)
            train_correct += pred.eq(labels).sum().item()

        epoch_train_acc = 100. * train_correct / train_total

        model.eval()
        
        val_loss, strict_correct, relaxed_correct, base_sign_correct, val_total = 0.0, 0, 0, 0, 0
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(dml, non_blocking=True)
                labels = labels.to(dml, non_blocking=True)
                
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                val_total += labels.size(0)
                
                _, top1_preds = outputs.max(1) 
                
                for i in range(labels.size(0)):
                    true_id = labels[i].item()
                    top1_id = top1_preds[i].item()
                    
                    true_name = id_to_class[true_id]
                    pred_name = id_to_class[top1_id]
                    
                    if top1_id == true_id:
                        strict_correct += 1
                        relaxed_correct += 1
                    else:
                        prediction_pair = set([true_id, top1_id])
                        if prediction_pair in allowed_confusions:
                            relaxed_correct += 1
                            
                    true_base = true_name.split('_')[0] if '_' in true_name else true_name
                    pred_base = pred_name.split('_')[0] if '_' in pred_name else pred_name
                    
                    if true_base == pred_base or set([true_base, pred_base]) in base_allowed_confusions:
                        base_sign_correct += 1
                
        strict_acc = 100. * strict_correct / val_total
        relaxed_acc = 100. * relaxed_correct / val_total
        base_sign_acc = 100. * base_sign_correct / val_total 

        history_train.append(epoch_train_acc)
        history_strict.append(strict_acc)
        history_relaxed.append(relaxed_acc)
        history_base.append(base_sign_acc)

        print(f"Epoch [{epoch+1:02d}/{EPOCHS}] | Train: {epoch_train_acc:.2f}% | "
              f"Val Strict: {strict_acc:.2f}% | Relaxed: {relaxed_acc:.2f}% -> Base Sign: {base_sign_acc:.2f}%")

        if relaxed_acc > best_val_acc: 
            best_val_acc = relaxed_acc
            save_path = os.path.join(MODELS_DIR, "expert_swish_9frame.pth")
            torch.save(model.state_dict(), save_path)
            
        scheduler.step()

    print("\n" + "="*60)
    print(f"🎉 Training Complete! Best Relaxed Val Accuracy: {best_val_acc:.2f}%")
    print("="*60)

    # ========================================================
    # 5. GRAPH GENERATION LOGIC (UPDATED PATHS)
    # ========================================================
    print(f"⏳ Generating Detailed Accuracy Graph in {MODELS_DIR}...")
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, EPOCHS + 1), history_train, label='Train Accuracy', color='blue')
    plt.plot(range(1, EPOCHS + 1), history_strict, label='Simple Val Acc', color='red')
    plt.plot(range(1, EPOCHS + 1), history_relaxed, label='Val Relaxed', color='green')
    plt.plot(range(1, EPOCHS + 1), history_base, label='Val Base Sign', color='orange', linestyle='--')
    
    plt.title('Expert Swish Model Accuracy (Detailed Metrics)')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True)
    
    # 🟢 Use os.path.join with MODELS_DIR
    graph_path_detailed = os.path.join(MODELS_DIR, "accuracy_graph_expert_swish_detailed.png")
    plt.savefig(graph_path_detailed)
    plt.close()
    print(f"✅ Saved Detailed Graph to {graph_path_detailed}")

    print(f"⏳ Generating Traditional Accuracy Graph in {MODELS_DIR}...")
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, EPOCHS + 1), history_train, label='Train Accuracy', color='blue')
    plt.plot(range(1, EPOCHS + 1), history_strict, label='Val Accuracy', color='red') 
    
    plt.title('Expert Swish Model Accuracy (Train vs Validation)')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True)
    
    # 🟢 Use os.path.join with MODELS_DIR
    graph_path_traditional = os.path.join(MODELS_DIR, "accuracy_graph_expert_swish_traditional.png")
    plt.savefig(graph_path_traditional)
    plt.close()
    print(f"✅ Saved Traditional Graph to {graph_path_traditional}")

if __name__ == "__main__":
    train_baseline_model()


🚀 SWISH ACTIVATION STUDY STARTED. Using GPU: AMD Radeon RX 7900 GRE 
🔒 Random Seed locked to 42 for strict reproducibility.
📂 Booting RAM-Cache Loader for train...
⏳ Loading arrays into RAM (This takes ~1 min but makes epochs 10x faster)...
✅ Successfully cached 34200 tensors in Memory!
📂 Booting RAM-Cache Loader for val...
⏳ Loading arrays into RAM (This takes ~1 min but makes epochs 10x faster)...
✅ Successfully cached 8681 tensors in Memory!
Epoch [01/50] | Train: 10.59% | Val Strict: 22.20% | Relaxed: 22.20% -> Base Sign: 23.22%
Epoch [02/50] | Train: 36.35% | Val Strict: 44.35% | Relaxed: 44.38% -> Base Sign: 45.84%
Epoch [03/50] | Train: 56.61% | Val Strict: 48.40% | Relaxed: 48.47% -> Base Sign: 49.78%
Epoch [04/50] | Train: 69.12% | Val Strict: 58.44% | Relaxed: 58.53% -> Base Sign: 59.54%
Epoch [05/50] | Train: 77.37% | Val Strict: 69.09% | Relaxed: 69.30% -> Base Sign: 70.12%
Epoch [06/50] | Train: 82.93% | Val Strict: 69.85% | Relaxed: 70.31% -> Base Sign: 70.91%
Epoch [07/

Gelu function model

In [2]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torch_directml
import warnings
import matplotlib.pyplot as plt

# 🟢 Silence the harmless DirectML warning
warnings.filterwarnings("ignore", message=".*aten::lerp.Scalar_out.*")

# ==========================================
# 1. CORE DATA STRUCTURES & MODEL
# ==========================================
class Graph:
    def __init__(self):
        self.num_node = 68  
        self.edges = self._get_edges()
        self.A = self._get_adjacency_matrix()

    def _get_edges(self):
        pose_edges = [
            (0,1), (1,2), (2,3), (3,7),
            (0,4), (4,5), (5,6), (6,8),
            (9,10),
            (11,12), (11,23), (12,24), (23,24),
            (11,13), (13,15),
            (12,14), (14,16),
            (15,17), (15,19), (15,21),
            (16,18), (16,20), (16,22)
        ]
        face_edges = [(0, 25)]
        hand_links = [(0,1), (1,2), (2,3), (3,4),
                      (0,5), (5,6), (6,7), (7,8),
                      (5,9), (9,10), (10,11), (11,12),
                      (9,13), (13,14), (14,15), (15,16),
                      (13,17), (0,17), (17,18), (18,19), (19,20)]
        left_hand_edges = [(s + 26, e + 26) for s, e in hand_links]
        right_hand_edges = [(s + 47, e + 47) for s, e in hand_links]
        connection_edges = [(15, 26), (16, 47)]
        
        return pose_edges + face_edges + left_hand_edges + right_hand_edges + connection_edges

    def _get_adjacency_matrix(self):
        A = np.zeros((self.num_node, self.num_node))
        for i, j in self.edges:
            A[i, j] = 1; A[j, i] = 1
        return torch.tensor(A, dtype=torch.float32)

class CleanBanglaDataset(Dataset):
    def __init__(self, split_dir, class_to_id):
        self.samples = []
        
        print(f"📂 Booting RAM-Cache Loader for {os.path.basename(split_dir)}...")
        print(f"⏳ Loading arrays into RAM (This takes ~1 min but makes epochs 10x faster)...")
        
        for class_name in os.listdir(split_dir):
            class_path = os.path.join(split_dir, class_name)
            if not os.path.isdir(class_path): continue
            
            if class_name in class_to_id:
                cid = class_to_id[class_name]
                for f_name in os.listdir(class_path):
                    if f_name.endswith('.npy'):
                        f_path = os.path.join(class_path, f_name)
                        
                        raw_data = np.load(f_path).reshape(90, 68, 3) 
                        raw_data = raw_data - np.mean(raw_data, axis=1, keepdims=True) 
                        raw_data = raw_data.transpose(2, 0, 1)
                        self.samples.append((raw_data, cid))
                        
        print(f"✅ Successfully cached {len(self.samples)} tensors in Memory!")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        data_cached, label = self.samples[idx]
        data = data_cached.copy()
        return torch.tensor(data, dtype=torch.float32), torch.tensor(label, dtype=torch.long)

class SpatialGraphConv(nn.Module):
    def __init__(self, in_c, out_c, A):
        super().__init__()
        self.register_buffer('A', A)
        self.conv = nn.Conv2d(in_c, out_c, 1)
    def forward(self, x):
        x = torch.einsum('nctv,vw->nctw', (x, self.A))
        return self.conv(x)

class STGCN_Block(nn.Module):
    def __init__(self, in_c, out_c, A, stride=1, dropout=0.4):
        super().__init__()
        self.sgcn = SpatialGraphConv(in_c, out_c, A)
        self.tgcn = nn.Sequential(
            nn.BatchNorm2d(out_c), 
            nn.GELU(), # 🟢 CHANGED: GELU Activation
            nn.Conv2d(out_c, out_c, (9, 1), (stride, 1), (4, 0)),
            nn.BatchNorm2d(out_c), 
            nn.Dropout(dropout)
        )
        self.res = nn.Sequential(nn.Conv2d(in_c, out_c, 1, (stride, 1)), nn.BatchNorm2d(out_c)) if in_c != out_c or stride != 1 else nn.Identity()
        
    def forward(self, x): 
        # 🟢 CHANGED: GELU applied to residual output
        return F.gelu(self.tgcn(self.sgcn(x)) + self.res(x)) 

class BanglaSignSTGCN(nn.Module):
    def __init__(self, num_classes, A, dropout_rate=0.4):
        super().__init__()
        self.layer1 = STGCN_Block(3, 64, A, dropout=dropout_rate)
        self.layer2 = STGCN_Block(64, 64, A, dropout=dropout_rate)
        self.layer3 = STGCN_Block(64, 64, A, dropout=dropout_rate)
        self.layer4 = STGCN_Block(64, 128, A, stride=2, dropout=dropout_rate)
        self.layer5 = STGCN_Block(128, 128, A, dropout=dropout_rate)
        self.layer6 = STGCN_Block(128, 128, A, dropout=dropout_rate)
        self.layer7 = STGCN_Block(128, 256, A, stride=2, dropout=dropout_rate)
        self.layer8 = STGCN_Block(256, 256, A, dropout=dropout_rate)
        self.layer9 = STGCN_Block(256, 256, A, dropout=dropout_rate)
        self.fcn = nn.Conv2d(256, num_classes, 1)
    def forward(self, x):
        for l in [self.layer1,self.layer2,self.layer3,self.layer4,self.layer5,self.layer6,self.layer7,self.layer8,self.layer9]: x = l(x)
        x = F.avg_pool2d(x, x.size()[2:])
        return self.fcn(x).view(x.size(0), -1)

# ==========================================
# 2. EXPERT TRAINING ENGINE (GELU ABLATION)
# ==========================================
def train_baseline_model():
    # 🟢 REPRODUCIBILITY LOCK
    seed = 42
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    DATASET_DIR = r"C:\Users\User\Documents\Personal Akams\Thesis\Final_Thesis_Dataset_1\Fold_1"
    TRAIN_DIR = os.path.join(DATASET_DIR, "train")
    VAL_DIR = os.path.join(DATASET_DIR, "val")
    
    MODELS_DIR = "GELU Function Experiment"
    os.makedirs(MODELS_DIR, exist_ok=True)
    
    # 🟢 EXPERT ENGINE SETTINGS
    EPOCHS = 50
    BATCH_SIZE = 32
    LEARNING_RATE = 0.001 
    DROPOUT = 0.4
    WEIGHT_DECAY = 1e-4
    NOISE_LEVEL = 0.005 
    
    dml = torch_directml.device()
    print(f"\n🚀 GELU ACTIVATION STUDY STARTED. Using GPU: {torch_directml.device_name(0)}")
    print(f"🔒 Random Seed locked to {seed} for strict reproducibility.")

    all_classes = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
    class_to_id = {cls_name: idx for idx, cls_name in enumerate(all_classes)}
    id_to_class = {idx: cls_name for cls_name, idx in class_to_id.items()} 
    num_classes = len(all_classes)
    
    homophone_pairs = [
        ("0_Lefthand", "O_Lefthand"),
        ("0_Righthand", "O_Righthand"),
        ("2_Lefthand", "V_Lefthand"),
        ("2_Righthand", "V_Righthand")
    ]
    
    allowed_confusions = []
    for a, b in homophone_pairs:
        if a in class_to_id and b in class_to_id:
            allowed_confusions.append(set([class_to_id[a], class_to_id[b]]))

    base_allowed_confusions = []
    for a, b in homophone_pairs:
        base_a = a.split('_')[0] if '_' in a else a
        base_b = b.split('_')[0] if '_' in b else b
        pair = set([base_a, base_b])
        if pair not in base_allowed_confusions:
            base_allowed_confusions.append(pair)

    train_data = CleanBanglaDataset(TRAIN_DIR, class_to_id)
    val_data = CleanBanglaDataset(VAL_DIR, class_to_id)

    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    graph = Graph()
    model = BanglaSignSTGCN(num_classes, graph.A, dropout_rate=DROPOUT).to(dml)
    
    # 🟢 EXPERT ENGINE UPGRADES
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, foreach=False)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    best_val_acc = 0.0

    history_train = []
    history_strict = []
    history_relaxed = []
    history_base = []

    for epoch in range(EPOCHS):
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        
        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs = inputs.to(dml, non_blocking=True)
            labels = labels.to(dml, non_blocking=True)
            
            if NOISE_LEVEL > 0.0:
                noise = torch.randn(*inputs.shape, device=dml) * NOISE_LEVEL
                inputs = inputs + noise
                
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * inputs.size(0)
            _, pred = outputs.max(1)
            train_total += labels.size(0)
            train_correct += pred.eq(labels).sum().item()

        epoch_train_acc = 100. * train_correct / train_total

        model.eval()
        
        val_loss, strict_correct, relaxed_correct, base_sign_correct, val_total = 0.0, 0, 0, 0, 0
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(dml, non_blocking=True)
                labels = labels.to(dml, non_blocking=True)
                
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                val_total += labels.size(0)
                
                _, top1_preds = outputs.max(1) 
                
                for i in range(labels.size(0)):
                    true_id = labels[i].item()
                    top1_id = top1_preds[i].item()
                    
                    true_name = id_to_class[true_id]
                    pred_name = id_to_class[top1_id]
                    
                    if top1_id == true_id:
                        strict_correct += 1
                        relaxed_correct += 1
                    else:
                        prediction_pair = set([true_id, top1_id])
                        if prediction_pair in allowed_confusions:
                            relaxed_correct += 1
                            
                    true_base = true_name.split('_')[0] if '_' in true_name else true_name
                    pred_base = pred_name.split('_')[0] if '_' in pred_name else pred_name
                    
                    if true_base == pred_base or set([true_base, pred_base]) in base_allowed_confusions:
                        base_sign_correct += 1
                
        strict_acc = 100. * strict_correct / val_total
        relaxed_acc = 100. * relaxed_correct / val_total
        base_sign_acc = 100. * base_sign_correct / val_total 

        history_train.append(epoch_train_acc)
        history_strict.append(strict_acc)
        history_relaxed.append(relaxed_acc)
        history_base.append(base_sign_acc)

        print(f"Epoch [{epoch+1:02d}/{EPOCHS}] | Train: {epoch_train_acc:.2f}% | "
              f"Val Strict: {strict_acc:.2f}% | Relaxed: {relaxed_acc:.2f}% -> Base Sign: {base_sign_acc:.2f}%")

        if relaxed_acc > best_val_acc: 
            best_val_acc = relaxed_acc
            save_path = os.path.join(MODELS_DIR, "expert_gelu_9frame.pth")
            torch.save(model.state_dict(), save_path)
            
        scheduler.step()

    print("\n" + "="*60)
    print(f"🎉 Training Complete! Best Relaxed Val Accuracy: {best_val_acc:.2f}%")
    print("="*60)

    print(f"⏳ Generating Detailed Accuracy Graph in {MODELS_DIR}...")
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, EPOCHS + 1), history_train, label='Train Accuracy', color='blue')
    plt.plot(range(1, EPOCHS + 1), history_strict, label='Simple Val Acc', color='red')
    plt.plot(range(1, EPOCHS + 1), history_relaxed, label='Val Relaxed', color='green')
    plt.plot(range(1, EPOCHS + 1), history_base, label='Val Base Sign', color='orange', linestyle='--')
    
    plt.title('Expert GELU Model Accuracy (Detailed Metrics)')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True)
    
    graph_path_detailed = os.path.join(MODELS_DIR, "accuracy_graph_expert_gelu_detailed.png")
    plt.savefig(graph_path_detailed)
    plt.close()
    print(f"✅ Saved Detailed Graph to {graph_path_detailed}")

    print(f"⏳ Generating Traditional Accuracy Graph in {MODELS_DIR}...")
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, EPOCHS + 1), history_train, label='Train Accuracy', color='blue')
    plt.plot(range(1, EPOCHS + 1), history_strict, label='Val Accuracy', color='red') 
    
    plt.title('Expert GELU Model Accuracy (Train vs Validation)')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True)
    
    graph_path_traditional = os.path.join(MODELS_DIR, "accuracy_graph_expert_gelu_traditional.png")
    plt.savefig(graph_path_traditional)
    plt.close()
    print(f"✅ Saved Traditional Graph to {graph_path_traditional}")

if __name__ == "__main__":
    train_baseline_model()


🚀 GELU ACTIVATION STUDY STARTED. Using GPU: AMD Radeon RX 7900 GRE 
🔒 Random Seed locked to 42 for strict reproducibility.
📂 Booting RAM-Cache Loader for train...
⏳ Loading arrays into RAM (This takes ~1 min but makes epochs 10x faster)...
✅ Successfully cached 34200 tensors in Memory!
📂 Booting RAM-Cache Loader for val...
⏳ Loading arrays into RAM (This takes ~1 min but makes epochs 10x faster)...
✅ Successfully cached 8681 tensors in Memory!
Epoch [01/50] | Train: 12.82% | Val Strict: 27.43% | Relaxed: 27.44% -> Base Sign: 28.88%
Epoch [02/50] | Train: 41.52% | Val Strict: 46.31% | Relaxed: 46.45% -> Base Sign: 47.76%
Epoch [03/50] | Train: 59.52% | Val Strict: 61.08% | Relaxed: 61.21% -> Base Sign: 62.26%
Epoch [04/50] | Train: 70.89% | Val Strict: 64.28% | Relaxed: 64.45% -> Base Sign: 65.44%
Epoch [05/50] | Train: 79.06% | Val Strict: 69.16% | Relaxed: 69.29% -> Base Sign: 70.08%
Epoch [06/50] | Train: 84.47% | Val Strict: 75.56% | Relaxed: 75.91% -> Base Sign: 76.43%
Epoch [07/5

9 layer+Attention layer

In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torch_directml
import warnings
import matplotlib.pyplot as plt

# 🟢 Silence the harmless DirectML warning
warnings.filterwarnings("ignore", message=".*aten::lerp.Scalar_out.*")

# ==========================================
# 1. CORE DATA STRUCTURES & MODEL
# ==========================================
class Graph:
    def __init__(self):
        self.num_node = 68  
        self.edges = self._get_edges()
        self.A = self._get_adjacency_matrix()

    def _get_edges(self):
        pose_edges = [
            (0,1), (1,2), (2,3), (3,7),
            (0,4), (4,5), (5,6), (6,8),
            (9,10),
            (11,12), (11,23), (12,24), (23,24),
            (11,13), (13,15),
            (12,14), (14,16),
            (15,17), (15,19), (15,21),
            (16,18), (16,20), (16,22)
        ]
        face_edges = [(0, 25)]
        hand_links = [(0,1), (1,2), (2,3), (3,4),
                      (0,5), (5,6), (6,7), (7,8),
                      (5,9), (9,10), (10,11), (11,12),
                      (9,13), (13,14), (14,15), (15,16),
                      (13,17), (0,17), (17,18), (18,19), (19,20)]
        left_hand_edges = [(s + 26, e + 26) for s, e in hand_links]
        right_hand_edges = [(s + 47, e + 47) for s, e in hand_links]
        connection_edges = [(15, 26), (16, 47)]
        
        return pose_edges + face_edges + left_hand_edges + right_hand_edges + connection_edges

    def _get_adjacency_matrix(self):
        A = np.zeros((self.num_node, self.num_node))
        for i, j in self.edges:
            A[i, j] = 1; A[j, i] = 1
        return torch.tensor(A, dtype=torch.float32)

class CleanBanglaDataset(Dataset):
    def __init__(self, split_dir, class_to_id):
        self.samples = []
        
        print(f"📂 Booting RAM-Cache Loader for {os.path.basename(split_dir)}...")
        
        for class_name in os.listdir(split_dir):
            class_path = os.path.join(split_dir, class_name)
            if not os.path.isdir(class_path): continue
            
            if class_name in class_to_id:
                cid = class_to_id[class_name]
                for f_name in os.listdir(class_path):
                    if f_name.endswith('.npy'):
                        f_path = os.path.join(class_path, f_name)
                        
                        raw_data = np.load(f_path).reshape(90, 68, 3) 
                        raw_data = raw_data - np.mean(raw_data, axis=1, keepdims=True) 
                        raw_data = raw_data.transpose(2, 0, 1)
                        
                        # ⚡ Pre-Tensorize data
                        tensor_data = torch.tensor(raw_data, dtype=torch.float32)
                        tensor_label = torch.tensor(cid, dtype=torch.long)
                        self.samples.append((tensor_data, tensor_label))
                        
        print(f"✅ Successfully cached {len(self.samples)} PyTorch Tensors in Memory!")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx): return self.samples[idx]

class SpatialGraphConv(nn.Module):
    def __init__(self, in_c, out_c, A):
        super().__init__()
        self.register_buffer('A', A)
        self.conv = nn.Conv2d(in_c, out_c, 1)
    def forward(self, x):
        x = torch.einsum('nctv,vw->nctw', (x, self.A))
        return self.conv(x)

# 🟢 NEW: Channel Attention Module (Squeeze-and-Excitation)
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=4):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction, in_channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class STGCN_Block(nn.Module):
    def __init__(self, in_c, out_c, A, stride=1, dropout=0.4):
        super().__init__()
        self.sgcn = SpatialGraphConv(in_c, out_c, A)
        self.tgcn = nn.Sequential(
            nn.BatchNorm2d(out_c), 
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, (9, 1), (stride, 1), (4, 0)),
            nn.BatchNorm2d(out_c), 
            nn.Dropout(dropout)
        )
        # 🟢 ADDED: Attention layer applied after temporal convolution
        self.attention = ChannelAttention(out_c)
        self.res = nn.Sequential(nn.Conv2d(in_c, out_c, 1, (stride, 1)), nn.BatchNorm2d(out_c)) if in_c != out_c or stride != 1 else nn.Identity()
        
    def forward(self, x): 
        # Pass through SGCN -> TGCN -> Attention -> Residual
        sgcn_out = self.sgcn(x)
        tgcn_out = self.tgcn(sgcn_out)
        attended_out = self.attention(tgcn_out)
        return F.relu(attended_out + self.res(x)) 

# 🔴 STANDARD 9-LAYER ARCHITECTURE
class BanglaSignSTGCN(nn.Module):
    def __init__(self, num_classes, A, dropout_rate=0.4):
        super().__init__()
        self.layer1 = STGCN_Block(3, 64, A, dropout=dropout_rate)
        self.layer2 = STGCN_Block(64, 64, A, dropout=dropout_rate)
        self.layer3 = STGCN_Block(64, 64, A, dropout=dropout_rate)
        self.layer4 = STGCN_Block(64, 128, A, stride=2, dropout=dropout_rate)
        self.layer5 = STGCN_Block(128, 128, A, dropout=dropout_rate)
        self.layer6 = STGCN_Block(128, 128, A, dropout=dropout_rate)
        self.layer7 = STGCN_Block(128, 256, A, stride=2, dropout=dropout_rate)
        self.layer8 = STGCN_Block(256, 256, A, dropout=dropout_rate)
        self.layer9 = STGCN_Block(256, 256, A, dropout=dropout_rate)
        self.fcn = nn.Conv2d(256, num_classes, 1)

    def forward(self, x):
        for l in [self.layer1,self.layer2,self.layer3,self.layer4,self.layer5,self.layer6,self.layer7,self.layer8,self.layer9]: x = l(x)
        x = F.avg_pool2d(x, x.size()[2:])
        return self.fcn(x).view(x.size(0), -1)

# ==========================================
# 2. TRAINING ENGINE (9-LAYER ATTENTION ABLATION)
# ==========================================
def train_attention_model():
    seed = 42
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    DATASET_DIR = r"C:\Users\User\Documents\Personal Akams\Thesis\Final_Thesis_Dataset_1\Fold_1"
    TRAIN_DIR = os.path.join(DATASET_DIR, "train")
    VAL_DIR = os.path.join(DATASET_DIR, "val")
    
    # 🟢 UPDATED: Directory for Attention Models
    MODELS_DIR = "7.9_layer_Attention_Models"
    os.makedirs(MODELS_DIR, exist_ok=True)
    
    EPOCHS = 50
    BATCH_SIZE = 32
    LEARNING_RATE = 0.001
    DROPOUT = 0.4
    WEIGHT_DECAY = 1e-4
    NOISE_LEVEL = 0.005 
    
    dml = torch_directml.device()
    print(f"\n🚀 9-LAYER ATTENTION ABLATION STARTED. Using GPU: {torch_directml.device_name(0)}")
    print(f"🔬 Testing: 9-Layer Architecture with SE Attention, ReLU, and 9-Frame Window.")
    print(f"🔒 Random Seed locked to {seed} for strict reproducibility.")

    all_classes = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
    class_to_id = {cls_name: idx for idx, cls_name in enumerate(all_classes)}
    id_to_class = {idx: cls_name for cls_name, idx in class_to_id.items()} 
    num_classes = len(all_classes)
    
    homophone_pairs = [
        ("0_Lefthand", "O_Lefthand"), ("0_Righthand", "O_Righthand"),
        ("2_Lefthand", "V_Lefthand"), ("2_Righthand", "V_Righthand")
    ]
    
    allowed_confusions = []
    for a, b in homophone_pairs:
        if a in class_to_id and b in class_to_id:
            allowed_confusions.append(set([class_to_id[a], class_to_id[b]]))

    base_allowed_confusions = []
    for a, b in homophone_pairs:
        base_a = a.split('_')[0] if '_' in a else a
        base_b = b.split('_')[0] if '_' in b else b
        pair = set([base_a, base_b])
        if pair not in base_allowed_confusions:
            base_allowed_confusions.append(pair)

    train_data = CleanBanglaDataset(TRAIN_DIR, class_to_id)
    val_data = CleanBanglaDataset(VAL_DIR, class_to_id)

    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=False)
    val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

    graph = Graph()
    model = BanglaSignSTGCN(num_classes, graph.A, dropout_rate=DROPOUT).to(dml)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, foreach=False)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    best_val_acc = 0.0

    history_train = []
    history_strict = []
    history_relaxed = []
    history_base = []

    for epoch in range(EPOCHS):
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        
        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs = inputs.to(dml, non_blocking=True)
            labels = labels.to(dml, non_blocking=True)
            
            if NOISE_LEVEL > 0.0:
                noise = torch.randn(*inputs.shape, device=dml) * NOISE_LEVEL
                inputs = inputs + noise
                
            optimizer.zero_grad(set_to_none=True)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * inputs.size(0)
            _, pred = outputs.max(1)
            train_total += labels.size(0)
            train_correct += pred.eq(labels).sum().item()

            if epoch == 0 and (batch_idx + 1) % 100 == 0:
                print(f"   ⏳ Epoch [{epoch+1}/{EPOCHS}] - Processing Batch {batch_idx+1}/{len(train_loader)}...")

        epoch_train_acc = 100. * train_correct / train_total

        model.eval()
        val_loss, strict_correct, relaxed_correct, base_sign_correct, val_total = 0.0, 0, 0, 0, 0
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(dml, non_blocking=True)
                labels = labels.to(dml, non_blocking=True)
                
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                val_total += labels.size(0)
                
                _, top1_preds = outputs.max(1) 
                
                for i in range(labels.size(0)):
                    true_id = labels[i].item()
                    top1_id = top1_preds[i].item()
                    
                    true_name = id_to_class[true_id]
                    pred_name = id_to_class[top1_id]
                    
                    if top1_id == true_id:
                        strict_correct += 1
                        relaxed_correct += 1
                    else:
                        prediction_pair = set([true_id, top1_id])
                        if prediction_pair in allowed_confusions:
                            relaxed_correct += 1
                            
                    true_base = true_name.split('_')[0] if '_' in true_name else true_name
                    pred_base = pred_name.split('_')[0] if '_' in pred_name else pred_name
                    
                    if true_base == pred_base or set([true_base, pred_base]) in base_allowed_confusions:
                        base_sign_correct += 1
                
        strict_acc = 100. * strict_correct / val_total
        relaxed_acc = 100. * relaxed_correct / val_total
        base_sign_acc = 100. * base_sign_correct / val_total 

        history_train.append(epoch_train_acc)
        history_strict.append(strict_acc)
        history_relaxed.append(relaxed_acc)
        history_base.append(base_sign_acc)

        print(f"Epoch [{epoch+1:02d}/{EPOCHS}] | Train: {epoch_train_acc:.2f}% | "
              f"Val Strict: {strict_acc:.2f}% | Relaxed: {relaxed_acc:.2f}% -> Base Sign: {base_sign_acc:.2f}%")

        if relaxed_acc > best_val_acc: 
            best_val_acc = relaxed_acc
            # 🟢 UPDATED: File save name
            save_path = os.path.join(MODELS_DIR, "9_layer_attention_relu_9frame.pth")
            torch.save(model.state_dict(), save_path)
            
        scheduler.step()

    print("\n" + "="*60)
    print(f"🎉 Training Complete! Best Relaxed Val Accuracy: {best_val_acc:.2f}%")
    print("="*60)

    print(f"⏳ Generating Detailed Accuracy Graph in {MODELS_DIR}...")
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, EPOCHS + 1), history_train, label='Train Accuracy', color='blue')
    plt.plot(range(1, EPOCHS + 1), history_strict, label='Simple Val Acc', color='red')
    plt.plot(range(1, EPOCHS + 1), history_relaxed, label='Val Relaxed', color='green')
    plt.plot(range(1, EPOCHS + 1), history_base, label='Val Base Sign', color='orange', linestyle='--')
    
    plt.title('9-Layer Attention ST-GCN Accuracy (Detailed Metrics)')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True)
    
    graph_path_detailed = os.path.join(MODELS_DIR, "accuracy_graph_9_layer_attention_detailed.png")
    plt.savefig(graph_path_detailed)
    plt.close()
    print(f"✅ Saved Detailed Graph to {graph_path_detailed}")

    print(f"⏳ Generating Traditional Accuracy Graph in {MODELS_DIR}...")
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, EPOCHS + 1), history_train, label='Train Accuracy', color='blue')
    plt.plot(range(1, EPOCHS + 1), history_strict, label='Val Accuracy', color='red') 
    
    plt.title('9-Layer Attention ST-GCN Accuracy (Train vs Validation)')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True)
    
    graph_path_traditional = os.path.join(MODELS_DIR, "accuracy_graph_9_layer_attention_traditional.png")
    plt.savefig(graph_path_traditional)
    plt.close()
    print(f"✅ Saved Traditional Graph to {graph_path_traditional}")

if __name__ == "__main__":
    import multiprocessing
    multiprocessing.freeze_support()
    train_attention_model()


🚀 9-LAYER ATTENTION ABLATION STARTED. Using GPU: AMD Radeon RX 7900 GRE 
🔬 Testing: 9-Layer Architecture with SE Attention, ReLU, and 9-Frame Window.
🔒 Random Seed locked to 42 for strict reproducibility.
📂 Booting RAM-Cache Loader for train...
✅ Successfully cached 34200 PyTorch Tensors in Memory!
📂 Booting RAM-Cache Loader for val...
✅ Successfully cached 8681 PyTorch Tensors in Memory!
   ⏳ Epoch [1/50] - Processing Batch 100/1069...
   ⏳ Epoch [1/50] - Processing Batch 200/1069...
   ⏳ Epoch [1/50] - Processing Batch 300/1069...
   ⏳ Epoch [1/50] - Processing Batch 400/1069...
   ⏳ Epoch [1/50] - Processing Batch 500/1069...
   ⏳ Epoch [1/50] - Processing Batch 600/1069...
   ⏳ Epoch [1/50] - Processing Batch 700/1069...
   ⏳ Epoch [1/50] - Processing Batch 800/1069...
   ⏳ Epoch [1/50] - Processing Batch 900/1069...
   ⏳ Epoch [1/50] - Processing Batch 1000/1069...
Epoch [01/50] | Train: 14.58% | Val Strict: 32.32% | Relaxed: 32.33% -> Base Sign: 33.77%
Epoch [02/50] | Train: 45.

Depth 9	Temporal Window  9	Act Function ReLU	Attention Mechanism Channel Attention (SE)	
Start Learning Rate 0.001	LR Scheduling Cosine Annealing	
Dropout Rate 0.5	Noise Added 0.005	Label Smoothing 0.1

In [2]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torch_directml
import warnings
import matplotlib.pyplot as plt

# 🟢 Silence the harmless DirectML warning
warnings.filterwarnings("ignore", message=".*aten::lerp.Scalar_out.*")

# ==========================================
# 1. CORE DATA STRUCTURES & MODEL
# ==========================================
class Graph:
    def __init__(self):
        self.num_node = 68  
        self.edges = self._get_edges()
        self.A = self._get_adjacency_matrix()

    def _get_edges(self):
        pose_edges = [
            (0,1), (1,2), (2,3), (3,7),
            (0,4), (4,5), (5,6), (6,8),
            (9,10),
            (11,12), (11,23), (12,24), (23,24),
            (11,13), (13,15),
            (12,14), (14,16),
            (15,17), (15,19), (15,21),
            (16,18), (16,20), (16,22)
        ]
        face_edges = [(0, 25)]
        hand_links = [(0,1), (1,2), (2,3), (3,4),
                      (0,5), (5,6), (6,7), (7,8),
                      (5,9), (9,10), (10,11), (11,12),
                      (9,13), (13,14), (14,15), (15,16),
                      (13,17), (0,17), (17,18), (18,19), (19,20)]
        left_hand_edges = [(s + 26, e + 26) for s, e in hand_links]
        right_hand_edges = [(s + 47, e + 47) for s, e in hand_links]
        connection_edges = [(15, 26), (16, 47)]
        
        return pose_edges + face_edges + left_hand_edges + right_hand_edges + connection_edges

    def _get_adjacency_matrix(self):
        A = np.zeros((self.num_node, self.num_node))
        for i, j in self.edges:
            A[i, j] = 1; A[j, i] = 1
        return torch.tensor(A, dtype=torch.float32)

class CleanBanglaDataset(Dataset):
    def __init__(self, split_dir, class_to_id):
        self.samples = []
        
        print(f"📂 Booting RAM-Cache Loader for {os.path.basename(split_dir)}...")
        
        for class_name in os.listdir(split_dir):
            class_path = os.path.join(split_dir, class_name)
            if not os.path.isdir(class_path): continue
            
            if class_name in class_to_id:
                cid = class_to_id[class_name]
                for f_name in os.listdir(class_path):
                    if f_name.endswith('.npy'):
                        f_path = os.path.join(class_path, f_name)
                        
                        raw_data = np.load(f_path).reshape(90, 68, 3) 
                        raw_data = raw_data - np.mean(raw_data, axis=1, keepdims=True) 
                        raw_data = raw_data.transpose(2, 0, 1)
                        
                        # ⚡ Pre-Tensorize data
                        tensor_data = torch.tensor(raw_data, dtype=torch.float32)
                        tensor_label = torch.tensor(cid, dtype=torch.long)
                        self.samples.append((tensor_data, tensor_label))
                        
        print(f"✅ Successfully cached {len(self.samples)} PyTorch Tensors in Memory!")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx): return self.samples[idx]

class SpatialGraphConv(nn.Module):
    def __init__(self, in_c, out_c, A):
        super().__init__()
        self.register_buffer('A', A)
        self.conv = nn.Conv2d(in_c, out_c, 1)
    def forward(self, x):
        x = torch.einsum('nctv,vw->nctw', (x, self.A))
        return self.conv(x)

# 🟢 NEW: Channel Attention Module (Squeeze-and-Excitation)
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=4):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction, in_channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class STGCN_Block(nn.Module):
    def __init__(self, in_c, out_c, A, stride=1, dropout=0.4):
        super().__init__()
        self.sgcn = SpatialGraphConv(in_c, out_c, A)
        self.tgcn = nn.Sequential(
            nn.BatchNorm2d(out_c), 
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, (9, 1), (stride, 1), (4, 0)),
            nn.BatchNorm2d(out_c), 
            nn.Dropout(dropout)
        )
        # 🟢 ADDED: Attention layer applied after temporal convolution
        self.attention = ChannelAttention(out_c)
        self.res = nn.Sequential(nn.Conv2d(in_c, out_c, 1, (stride, 1)), nn.BatchNorm2d(out_c)) if in_c != out_c or stride != 1 else nn.Identity()
        
    def forward(self, x): 
        # Pass through SGCN -> TGCN -> Attention -> Residual
        sgcn_out = self.sgcn(x)
        tgcn_out = self.tgcn(sgcn_out)
        attended_out = self.attention(tgcn_out)
        return F.relu(attended_out + self.res(x)) 

# 🔴 STANDARD 9-LAYER ARCHITECTURE
class BanglaSignSTGCN(nn.Module):
    def __init__(self, num_classes, A, dropout_rate=0.5):
        super().__init__()
        self.layer1 = STGCN_Block(3, 64, A, dropout=dropout_rate)
        self.layer2 = STGCN_Block(64, 64, A, dropout=dropout_rate)
        self.layer3 = STGCN_Block(64, 64, A, dropout=dropout_rate)
        self.layer4 = STGCN_Block(64, 128, A, stride=2, dropout=dropout_rate)
        self.layer5 = STGCN_Block(128, 128, A, dropout=dropout_rate)
        self.layer6 = STGCN_Block(128, 128, A, dropout=dropout_rate)
        self.layer7 = STGCN_Block(128, 256, A, stride=2, dropout=dropout_rate)
        self.layer8 = STGCN_Block(256, 256, A, dropout=dropout_rate)
        self.layer9 = STGCN_Block(256, 256, A, dropout=dropout_rate)
        self.fcn = nn.Conv2d(256, num_classes, 1)

    def forward(self, x):
        for l in [self.layer1,self.layer2,self.layer3,self.layer4,self.layer5,self.layer6,self.layer7,self.layer8,self.layer9]: x = l(x)
        x = F.avg_pool2d(x, x.size()[2:])
        return self.fcn(x).view(x.size(0), -1)

# ==========================================
# 2. TRAINING ENGINE (9-LAYER ATTENTION ABLATION)
# ==========================================
def train_attention_model():
    seed = 42
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    DATASET_DIR = r"C:\Users\User\Documents\Personal Akams\Thesis\Final_Thesis_Dataset_1\Fold_1"
    TRAIN_DIR = os.path.join(DATASET_DIR, "train")
    VAL_DIR = os.path.join(DATASET_DIR, "val")
    
    # 🟢 UPDATED: Directory for Attention Models
    MODELS_DIR = "Depth 9_Temporal Window  9_Act Function ReLU_Attention Mechanism Channel Attention (SE)_Start Learning Rate 0.001_LR Scheduling Cosine Annealing_Dropout Rate 0.5_Noise Added 0.005_Label Smoothing 0.1"
    os.makedirs(MODELS_DIR, exist_ok=True)
    
    EPOCHS = 50
    BATCH_SIZE = 32
    LEARNING_RATE = 0.001
    DROPOUT = 0.5
    WEIGHT_DECAY = 1e-4
    NOISE_LEVEL = 0.005 
    
    target_idx = 0
    print("\n🔍 Scanning available DirectML GPUs...")
    for i in range(torch_directml.device_count()):
        gpu_name = torch_directml.device_name(i)
        print(f"  Found Device {i}: {gpu_name}")
        if "7900" in gpu_name or "GRE" in gpu_name or "RX" in gpu_name: 
            target_idx = i

    dml = torch_directml.device(target_idx)
    print(f"\n🚀 9-LAYER ATTENTION ABLATION STARTED. Using GPU: {torch_directml.device_name(target_idx)}")
    print(f"🔬 Testing: 9-Layer Architecture with SE Attention, ReLU, and 9-Frame Window.")
    print(f"🔒 Random Seed locked to {seed} for strict reproducibility.")

    all_classes = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
    class_to_id = {cls_name: idx for idx, cls_name in enumerate(all_classes)}
    id_to_class = {idx: cls_name for cls_name, idx in class_to_id.items()} 
    num_classes = len(all_classes)
    
    homophone_pairs = [
        ("0_Lefthand", "O_Lefthand"), ("0_Righthand", "O_Righthand"),
        ("2_Lefthand", "V_Lefthand"), ("2_Righthand", "V_Righthand")
    ]
    
    allowed_confusions = []
    for a, b in homophone_pairs:
        if a in class_to_id and b in class_to_id:
            allowed_confusions.append(set([class_to_id[a], class_to_id[b]]))

    base_allowed_confusions = []
    for a, b in homophone_pairs:
        base_a = a.split('_')[0] if '_' in a else a
        base_b = b.split('_')[0] if '_' in b else b
        pair = set([base_a, base_b])
        if pair not in base_allowed_confusions:
            base_allowed_confusions.append(pair)

    train_data = CleanBanglaDataset(TRAIN_DIR, class_to_id)
    val_data = CleanBanglaDataset(VAL_DIR, class_to_id)

    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=False)
    val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

    graph = Graph()
    model = BanglaSignSTGCN(num_classes, graph.A, dropout_rate=DROPOUT).to(dml)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, foreach=False)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    best_val_acc = 0.0

    history_train = []
    history_strict = []
    history_relaxed = []
    history_base = []

    for epoch in range(EPOCHS):
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        
        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs = inputs.to(dml, non_blocking=True)
            labels = labels.to(dml, non_blocking=True)
            
            if NOISE_LEVEL > 0.0:
                noise = torch.randn(*inputs.shape, device=dml) * NOISE_LEVEL
                inputs = inputs + noise
                
            optimizer.zero_grad(set_to_none=True)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * inputs.size(0)
            _, pred = outputs.max(1)
            train_total += labels.size(0)
            train_correct += pred.eq(labels).sum().item()

            if epoch == 0 and (batch_idx + 1) % 100 == 0:
                print(f"   ⏳ Epoch [{epoch+1}/{EPOCHS}] - Processing Batch {batch_idx+1}/{len(train_loader)}...")

        epoch_train_acc = 100. * train_correct / train_total

        model.eval()
        val_loss, strict_correct, relaxed_correct, base_sign_correct, val_total = 0.0, 0, 0, 0, 0
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(dml, non_blocking=True)
                labels = labels.to(dml, non_blocking=True)
                
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                val_total += labels.size(0)
                
                _, top1_preds = outputs.max(1) 
                
                for i in range(labels.size(0)):
                    true_id = labels[i].item()
                    top1_id = top1_preds[i].item()
                    
                    true_name = id_to_class[true_id]
                    pred_name = id_to_class[top1_id]
                    
                    if top1_id == true_id:
                        strict_correct += 1
                        relaxed_correct += 1
                    else:
                        prediction_pair = set([true_id, top1_id])
                        if prediction_pair in allowed_confusions:
                            relaxed_correct += 1
                            
                    true_base = true_name.split('_')[0] if '_' in true_name else true_name
                    pred_base = pred_name.split('_')[0] if '_' in pred_name else pred_name
                    
                    if true_base == pred_base or set([true_base, pred_base]) in base_allowed_confusions:
                        base_sign_correct += 1
                
        strict_acc = 100. * strict_correct / val_total
        relaxed_acc = 100. * relaxed_correct / val_total
        base_sign_acc = 100. * base_sign_correct / val_total 

        history_train.append(epoch_train_acc)
        history_strict.append(strict_acc)
        history_relaxed.append(relaxed_acc)
        history_base.append(base_sign_acc)

        print(f"Epoch [{epoch+1:02d}/{EPOCHS}] | Train: {epoch_train_acc:.2f}% | "
              f"Val Strict: {strict_acc:.2f}% | Relaxed: {relaxed_acc:.2f}% -> Base Sign: {base_sign_acc:.2f}%")

        if relaxed_acc > best_val_acc: 
            best_val_acc = relaxed_acc
            # 🟢 UPDATED: File save name
            save_path = os.path.join(MODELS_DIR, "9_layer_attention_relu_9frame.pth")
            torch.save(model.state_dict(), save_path)
            
        scheduler.step()

    print("\n" + "="*60)
    print(f"🎉 Training Complete! Best Relaxed Val Accuracy: {best_val_acc:.2f}%")
    print("="*60)

    print(f"⏳ Generating Detailed Accuracy Graph in {MODELS_DIR}...")
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, EPOCHS + 1), history_train, label='Train Accuracy', color='blue')
    plt.plot(range(1, EPOCHS + 1), history_strict, label='Simple Val Acc', color='red')
    plt.plot(range(1, EPOCHS + 1), history_relaxed, label='Val Relaxed', color='green')
    plt.plot(range(1, EPOCHS + 1), history_base, label='Val Base Sign', color='orange', linestyle='--')
    
    plt.title('9-Layer Attention ST-GCN Accuracy (Detailed Metrics)')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True)
    
    graph_path_detailed = os.path.join(MODELS_DIR, "accuracy_graph_9_layer_attention_detailed.png")
    plt.savefig(graph_path_detailed)
    plt.close()
    print(f"✅ Saved Detailed Graph to {graph_path_detailed}")

    print(f"⏳ Generating Traditional Accuracy Graph in {MODELS_DIR}...")
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, EPOCHS + 1), history_train, label='Train Accuracy', color='blue')
    plt.plot(range(1, EPOCHS + 1), history_strict, label='Val Accuracy', color='red') 
    
    plt.title('9-Layer Attention ST-GCN Accuracy (Train vs Validation)')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True)
    
    graph_path_traditional = os.path.join(MODELS_DIR, "accuracy_graph_9_layer_attention_traditional.png")
    plt.savefig(graph_path_traditional)
    plt.close()
    print(f"✅ Saved Traditional Graph to {graph_path_traditional}")

if __name__ == "__main__":
    import multiprocessing
    multiprocessing.freeze_support()
    train_attention_model()


🔍 Scanning available DirectML GPUs...
  Found Device 0: AMD Radeon(TM) Graphics 
  Found Device 1: AMD Radeon RX 7900 GRE 

🚀 9-LAYER ATTENTION ABLATION STARTED. Using GPU: AMD Radeon RX 7900 GRE 
🔬 Testing: 9-Layer Architecture with SE Attention, ReLU, and 9-Frame Window.
🔒 Random Seed locked to 42 for strict reproducibility.
📂 Booting RAM-Cache Loader for train...
✅ Successfully cached 34200 PyTorch Tensors in Memory!
📂 Booting RAM-Cache Loader for val...
✅ Successfully cached 8681 PyTorch Tensors in Memory!
   ⏳ Epoch [1/50] - Processing Batch 100/1069...
   ⏳ Epoch [1/50] - Processing Batch 200/1069...
   ⏳ Epoch [1/50] - Processing Batch 300/1069...
   ⏳ Epoch [1/50] - Processing Batch 400/1069...
   ⏳ Epoch [1/50] - Processing Batch 500/1069...
   ⏳ Epoch [1/50] - Processing Batch 600/1069...
   ⏳ Epoch [1/50] - Processing Batch 700/1069...
   ⏳ Epoch [1/50] - Processing Batch 800/1069...
   ⏳ Epoch [1/50] - Processing Batch 900/1069...
   ⏳ Epoch [1/50] - Processing Batch 1000/

Depth 9	Temporal Window  9	Act Function ReLU	Attention Mechanism Channel Attention (SE)	
Start Learning Rate 0.001	LR Scheduling Cosine Annealing	
Dropout Rate 0.4	Noise Added None	Label Smoothing 0.1

In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torch_directml
import warnings
import matplotlib.pyplot as plt

# 🟢 Silence the harmless DirectML warning
warnings.filterwarnings("ignore", message=".*aten::lerp.Scalar_out.*")

# ==========================================
# 1. CORE DATA STRUCTURES & MODEL
# ==========================================
class Graph:
    def __init__(self):
        self.num_node = 68  
        self.edges = self._get_edges()
        self.A = self._get_adjacency_matrix()

    def _get_edges(self):
        pose_edges = [
            (0,1), (1,2), (2,3), (3,7),
            (0,4), (4,5), (5,6), (6,8),
            (9,10),
            (11,12), (11,23), (12,24), (23,24),
            (11,13), (13,15),
            (12,14), (14,16),
            (15,17), (15,19), (15,21),
            (16,18), (16,20), (16,22)
        ]
        face_edges = [(0, 25)]
        hand_links = [(0,1), (1,2), (2,3), (3,4),
                      (0,5), (5,6), (6,7), (7,8),
                      (5,9), (9,10), (10,11), (11,12),
                      (9,13), (13,14), (14,15), (15,16),
                      (13,17), (0,17), (17,18), (18,19), (19,20)]
        left_hand_edges = [(s + 26, e + 26) for s, e in hand_links]
        right_hand_edges = [(s + 47, e + 47) for s, e in hand_links]
        connection_edges = [(15, 26), (16, 47)]
        
        return pose_edges + face_edges + left_hand_edges + right_hand_edges + connection_edges

    def _get_adjacency_matrix(self):
        A = np.zeros((self.num_node, self.num_node))
        for i, j in self.edges:
            A[i, j] = 1; A[j, i] = 1
        return torch.tensor(A, dtype=torch.float32)

class CleanBanglaDataset(Dataset):
    def __init__(self, split_dir, class_to_id):
        self.samples = []
        
        print(f"📂 Booting RAM-Cache Loader for {os.path.basename(split_dir)}...")
        
        for class_name in os.listdir(split_dir):
            class_path = os.path.join(split_dir, class_name)
            if not os.path.isdir(class_path): continue
            
            if class_name in class_to_id:
                cid = class_to_id[class_name]
                for f_name in os.listdir(class_path):
                    if f_name.endswith('.npy'):
                        f_path = os.path.join(class_path, f_name)
                        
                        raw_data = np.load(f_path).reshape(90, 68, 3) 
                        raw_data = raw_data - np.mean(raw_data, axis=1, keepdims=True) 
                        raw_data = raw_data.transpose(2, 0, 1)
                        
                        # ⚡ Pre-Tensorize data
                        tensor_data = torch.tensor(raw_data, dtype=torch.float32)
                        tensor_label = torch.tensor(cid, dtype=torch.long)
                        self.samples.append((tensor_data, tensor_label))
                        
        print(f"✅ Successfully cached {len(self.samples)} PyTorch Tensors in Memory!")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx): return self.samples[idx]

class SpatialGraphConv(nn.Module):
    def __init__(self, in_c, out_c, A):
        super().__init__()
        self.register_buffer('A', A)
        self.conv = nn.Conv2d(in_c, out_c, 1)
    def forward(self, x):
        x = torch.einsum('nctv,vw->nctw', (x, self.A))
        return self.conv(x)

# 🟢 Channel Attention Module (Squeeze-and-Excitation)
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=4):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction, in_channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class STGCN_Block(nn.Module):
    def __init__(self, in_c, out_c, A, stride=1, dropout=0.4):
        super().__init__()
        self.sgcn = SpatialGraphConv(in_c, out_c, A)
        self.tgcn = nn.Sequential(
            nn.BatchNorm2d(out_c), 
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, (9, 1), (stride, 1), (4, 0)),
            nn.BatchNorm2d(out_c), 
            nn.Dropout(dropout)
        )
        # 🟢 Attention layer applied after temporal convolution
        self.attention = ChannelAttention(out_c)
        self.res = nn.Sequential(nn.Conv2d(in_c, out_c, 1, (stride, 1)), nn.BatchNorm2d(out_c)) if in_c != out_c or stride != 1 else nn.Identity()
        
    def forward(self, x): 
        # Pass through SGCN -> TGCN -> Attention -> Residual
        sgcn_out = self.sgcn(x)
        tgcn_out = self.tgcn(sgcn_out)
        attended_out = self.attention(tgcn_out)
        return F.relu(attended_out + self.res(x)) 

# 🔴 STANDARD 9-LAYER ARCHITECTURE
class BanglaSignSTGCN(nn.Module):
    def __init__(self, num_classes, A, dropout_rate=0.4):
        super().__init__()
        self.layer1 = STGCN_Block(3, 64, A, dropout=dropout_rate)
        self.layer2 = STGCN_Block(64, 64, A, dropout=dropout_rate)
        self.layer3 = STGCN_Block(64, 64, A, dropout=dropout_rate)
        self.layer4 = STGCN_Block(64, 128, A, stride=2, dropout=dropout_rate)
        self.layer5 = STGCN_Block(128, 128, A, dropout=dropout_rate)
        self.layer6 = STGCN_Block(128, 128, A, dropout=dropout_rate)
        self.layer7 = STGCN_Block(128, 256, A, stride=2, dropout=dropout_rate)
        self.layer8 = STGCN_Block(256, 256, A, dropout=dropout_rate)
        self.layer9 = STGCN_Block(256, 256, A, dropout=dropout_rate)
        self.fcn = nn.Conv2d(256, num_classes, 1)

    def forward(self, x):
        for l in [self.layer1,self.layer2,self.layer3,self.layer4,self.layer5,self.layer6,self.layer7,self.layer8,self.layer9]: x = l(x)
        x = F.avg_pool2d(x, x.size()[2:])
        return self.fcn(x).view(x.size(0), -1)

# ==========================================
# 2. TRAINING ENGINE 
# ==========================================
def train_attention_model():
    seed = 42
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    DATASET_DIR = r"C:\Users\User\Documents\Personal Akams\Thesis\Final_Thesis_Dataset_1\Fold_1"
    TRAIN_DIR = os.path.join(DATASET_DIR, "train")
    VAL_DIR = os.path.join(DATASET_DIR, "val")
    
    # 🟢 Descriptive Folder Name for the Ablation
    MODELS_DIR = "Ablation_9Layer_SE_NoNoise_Drop0.4"
    os.makedirs(MODELS_DIR, exist_ok=True)
    
    EPOCHS = 50
    BATCH_SIZE = 32
    LEARNING_RATE = 0.001
    DROPOUT = 0.4
    WEIGHT_DECAY = 1e-4
    NOISE_LEVEL = 0.0  # 🟢 SET TO 0.0 (None)
    
    # 🟢 THE GPU SCANNER FIX
    target_idx = 0
    print("\n🔍 Scanning available DirectML GPUs...")
    for i in range(torch_directml.device_count()):
        gpu_name = torch_directml.device_name(i)
        print(f"  Found Device {i}: {gpu_name}")
        if "7900" in gpu_name or "GRE" in gpu_name or "RX" in gpu_name: 
            target_idx = i

    dml = torch_directml.device(target_idx)
    print(f"\n🚀 EXPERIMENT STARTED. Locked onto: {torch_directml.device_name(target_idx)}")
    print(f"🔬 Configuration: 9-Layer, SE Attention, Drop 0.4, Noise 0.0, LR 0.001")
    print(f"🔒 Random Seed locked to {seed} for strict reproducibility.")

    all_classes = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
    class_to_id = {cls_name: idx for idx, cls_name in enumerate(all_classes)}
    id_to_class = {idx: cls_name for cls_name, idx in class_to_id.items()} 
    num_classes = len(all_classes)
    
    homophone_pairs = [
        ("0_Lefthand", "O_Lefthand"), ("0_Righthand", "O_Righthand"),
        ("2_Lefthand", "V_Lefthand"), ("2_Righthand", "V_Righthand")
    ]
    
    allowed_confusions = []
    for a, b in homophone_pairs:
        if a in class_to_id and b in class_to_id:
            allowed_confusions.append(set([class_to_id[a], class_to_id[b]]))

    base_allowed_confusions = []
    for a, b in homophone_pairs:
        base_a = a.split('_')[0] if '_' in a else a
        base_b = b.split('_')[0] if '_' in b else b
        pair = set([base_a, base_b])
        if pair not in base_allowed_confusions:
            base_allowed_confusions.append(pair)

    train_data = CleanBanglaDataset(TRAIN_DIR, class_to_id)
    val_data = CleanBanglaDataset(VAL_DIR, class_to_id)

    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=False)
    val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

    graph = Graph()
    model = BanglaSignSTGCN(num_classes, graph.A, dropout_rate=DROPOUT).to(dml)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, foreach=False)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    best_val_acc = 0.0

    history_train = []
    history_strict = []
    history_relaxed = []
    history_base = []

    for epoch in range(EPOCHS):
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        
        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs = inputs.to(dml, non_blocking=True)
            labels = labels.to(dml, non_blocking=True)
            
            # 🟢 Logic remains, but skips execution since NOISE_LEVEL is 0.0
            if NOISE_LEVEL > 0.0:
                noise = torch.randn(*inputs.shape, device=dml) * NOISE_LEVEL
                inputs = inputs + noise
                
            optimizer.zero_grad(set_to_none=True)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * inputs.size(0)
            _, pred = outputs.max(1)
            train_total += labels.size(0)
            train_correct += pred.eq(labels).sum().item()

            if epoch == 0 and (batch_idx + 1) % 100 == 0:
                print(f"   ⏳ Epoch [{epoch+1}/{EPOCHS}] - Processing Batch {batch_idx+1}/{len(train_loader)}...")

        epoch_train_acc = 100. * train_correct / train_total

        model.eval()
        val_loss, strict_correct, relaxed_correct, base_sign_correct, val_total = 0.0, 0, 0, 0, 0
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(dml, non_blocking=True)
                labels = labels.to(dml, non_blocking=True)
                
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                val_total += labels.size(0)
                
                _, top1_preds = outputs.max(1) 
                
                for i in range(labels.size(0)):
                    true_id = labels[i].item()
                    top1_id = top1_preds[i].item()
                    
                    true_name = id_to_class[true_id]
                    pred_name = id_to_class[top1_id]
                    
                    if top1_id == true_id:
                        strict_correct += 1
                        relaxed_correct += 1
                    else:
                        prediction_pair = set([true_id, top1_id])
                        if prediction_pair in allowed_confusions:
                            relaxed_correct += 1
                            
                    true_base = true_name.split('_')[0] if '_' in true_name else true_name
                    pred_base = pred_name.split('_')[0] if '_' in pred_name else pred_name
                    
                    if true_base == pred_base or set([true_base, pred_base]) in base_allowed_confusions:
                        base_sign_correct += 1
                
        strict_acc = 100. * strict_correct / val_total
        relaxed_acc = 100. * relaxed_correct / val_total
        base_sign_acc = 100. * base_sign_correct / val_total 

        history_train.append(epoch_train_acc)
        history_strict.append(strict_acc)
        history_relaxed.append(relaxed_acc)
        history_base.append(base_sign_acc)

        print(f"Epoch [{epoch+1:02d}/{EPOCHS}] | Train: {epoch_train_acc:.2f}% | "
              f"Val Strict: {strict_acc:.2f}% | Relaxed: {relaxed_acc:.2f}% -> Base Sign: {base_sign_acc:.2f}%")

        if relaxed_acc > best_val_acc: 
            best_val_acc = relaxed_acc
            save_path = os.path.join(MODELS_DIR, "9layer_SE_nonoise_drop0.4.pth")
            torch.save(model.state_dict(), save_path)
            
        scheduler.step()

    print("\n" + "="*60)
    print(f"🎉 Training Complete! Best Relaxed Val Accuracy: {best_val_acc:.2f}%")
    print("="*60)

    print(f"⏳ Generating Detailed Accuracy Graph in {MODELS_DIR}...")
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, EPOCHS + 1), history_train, label='Train Accuracy', color='blue')
    plt.plot(range(1, EPOCHS + 1), history_strict, label='Simple Val Acc', color='red')
    plt.plot(range(1, EPOCHS + 1), history_relaxed, label='Val Relaxed', color='green')
    plt.plot(range(1, EPOCHS + 1), history_base, label='Val Base Sign', color='orange', linestyle='--')
    
    plt.title('9-Layer SE (No Noise) ST-GCN Accuracy (Detailed)')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True)
    
    graph_path_detailed = os.path.join(MODELS_DIR, "accuracy_graph_detailed.png")
    plt.savefig(graph_path_detailed)
    plt.close()
    print(f"✅ Saved Detailed Graph to {graph_path_detailed}")

    print(f"⏳ Generating Traditional Accuracy Graph in {MODELS_DIR}...")
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, EPOCHS + 1), history_train, label='Train Accuracy', color='blue')
    plt.plot(range(1, EPOCHS + 1), history_strict, label='Val Accuracy', color='red') 
    
    plt.title('9-Layer SE (No Noise) ST-GCN Accuracy (Train vs Validation)')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True)
    
    graph_path_traditional = os.path.join(MODELS_DIR, "accuracy_graph_traditional.png")
    plt.savefig(graph_path_traditional)
    plt.close()
    print(f"✅ Saved Traditional Graph to {graph_path_traditional}")

if __name__ == "__main__":
    import multiprocessing
    multiprocessing.freeze_support()
    train_attention_model()


🔍 Scanning available DirectML GPUs...
  Found Device 0: AMD Radeon(TM) Graphics 
  Found Device 1: AMD Radeon RX 7900 GRE 

🚀 EXPERIMENT STARTED. Locked onto: AMD Radeon RX 7900 GRE 
🔬 Configuration: 9-Layer, SE Attention, Drop 0.4, Noise 0.0, LR 0.001
🔒 Random Seed locked to 42 for strict reproducibility.
📂 Booting RAM-Cache Loader for train...
✅ Successfully cached 34200 PyTorch Tensors in Memory!
📂 Booting RAM-Cache Loader for val...
✅ Successfully cached 8681 PyTorch Tensors in Memory!
   ⏳ Epoch [1/50] - Processing Batch 100/1069...
   ⏳ Epoch [1/50] - Processing Batch 200/1069...
   ⏳ Epoch [1/50] - Processing Batch 300/1069...
   ⏳ Epoch [1/50] - Processing Batch 400/1069...
   ⏳ Epoch [1/50] - Processing Batch 500/1069...
   ⏳ Epoch [1/50] - Processing Batch 600/1069...
   ⏳ Epoch [1/50] - Processing Batch 700/1069...
   ⏳ Epoch [1/50] - Processing Batch 800/1069...
   ⏳ Epoch [1/50] - Processing Batch 900/1069...
   ⏳ Epoch [1/50] - Processing Batch 1000/1069...
Epoch [01/50]

Depth 9	Temporal Window  9	Act Function ReLU	Attention Mechanism Channel Attention (SE)	
Start Learning Rate 0.001	LR Scheduling Cosine Annealing	
Dropout Rate 0.4	Noise Added 0.005	Label Smoothing None

In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torch_directml
import warnings
import matplotlib.pyplot as plt

# 🟢 Silence the harmless DirectML warning
warnings.filterwarnings("ignore", message=".*aten::lerp.Scalar_out.*")

# ==========================================
# 1. CORE DATA STRUCTURES & MODEL
# ==========================================
class Graph:
    def __init__(self):
        self.num_node = 68  
        self.edges = self._get_edges()
        self.A = self._get_adjacency_matrix()

    def _get_edges(self):
        pose_edges = [
            (0,1), (1,2), (2,3), (3,7),
            (0,4), (4,5), (5,6), (6,8),
            (9,10),
            (11,12), (11,23), (12,24), (23,24),
            (11,13), (13,15),
            (12,14), (14,16),
            (15,17), (15,19), (15,21),
            (16,18), (16,20), (16,22)
        ]
        face_edges = [(0, 25)]
        hand_links = [(0,1), (1,2), (2,3), (3,4),
                      (0,5), (5,6), (6,7), (7,8),
                      (5,9), (9,10), (10,11), (11,12),
                      (9,13), (13,14), (14,15), (15,16),
                      (13,17), (0,17), (17,18), (18,19), (19,20)]
        left_hand_edges = [(s + 26, e + 26) for s, e in hand_links]
        right_hand_edges = [(s + 47, e + 47) for s, e in hand_links]
        connection_edges = [(15, 26), (16, 47)]
        
        return pose_edges + face_edges + left_hand_edges + right_hand_edges + connection_edges

    def _get_adjacency_matrix(self):
        A = np.zeros((self.num_node, self.num_node))
        for i, j in self.edges:
            A[i, j] = 1; A[j, i] = 1
        return torch.tensor(A, dtype=torch.float32)

class CleanBanglaDataset(Dataset):
    def __init__(self, split_dir, class_to_id):
        self.samples = []
        
        print(f"📂 Booting RAM-Cache Loader for {os.path.basename(split_dir)}...")
        
        for class_name in os.listdir(split_dir):
            class_path = os.path.join(split_dir, class_name)
            if not os.path.isdir(class_path): continue
            
            if class_name in class_to_id:
                cid = class_to_id[class_name]
                for f_name in os.listdir(class_path):
                    if f_name.endswith('.npy'):
                        f_path = os.path.join(class_path, f_name)
                        
                        raw_data = np.load(f_path).reshape(90, 68, 3) 
                        raw_data = raw_data - np.mean(raw_data, axis=1, keepdims=True) 
                        raw_data = raw_data.transpose(2, 0, 1)
                        
                        # ⚡ Pre-Tensorize data
                        tensor_data = torch.tensor(raw_data, dtype=torch.float32)
                        tensor_label = torch.tensor(cid, dtype=torch.long)
                        self.samples.append((tensor_data, tensor_label))
                        
        print(f"✅ Successfully cached {len(self.samples)} PyTorch Tensors in Memory!")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx): return self.samples[idx]

class SpatialGraphConv(nn.Module):
    def __init__(self, in_c, out_c, A):
        super().__init__()
        self.register_buffer('A', A)
        self.conv = nn.Conv2d(in_c, out_c, 1)
    def forward(self, x):
        x = torch.einsum('nctv,vw->nctw', (x, self.A))
        return self.conv(x)

# 🟢 Channel Attention Module (Squeeze-and-Excitation)
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=4):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction, in_channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class STGCN_Block(nn.Module):
    def __init__(self, in_c, out_c, A, stride=1, dropout=0.4):
        super().__init__()
        self.sgcn = SpatialGraphConv(in_c, out_c, A)
        self.tgcn = nn.Sequential(
            nn.BatchNorm2d(out_c), 
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, (9, 1), (stride, 1), (4, 0)),
            nn.BatchNorm2d(out_c), 
            nn.Dropout(dropout)
        )
        # 🟢 Attention layer applied after temporal convolution
        self.attention = ChannelAttention(out_c)
        self.res = nn.Sequential(nn.Conv2d(in_c, out_c, 1, (stride, 1)), nn.BatchNorm2d(out_c)) if in_c != out_c or stride != 1 else nn.Identity()
        
    def forward(self, x): 
        # Pass through SGCN -> TGCN -> Attention -> Residual
        sgcn_out = self.sgcn(x)
        tgcn_out = self.tgcn(sgcn_out)
        attended_out = self.attention(tgcn_out)
        return F.relu(attended_out + self.res(x)) 

# 🔴 STANDARD 9-LAYER ARCHITECTURE
class BanglaSignSTGCN(nn.Module):
    def __init__(self, num_classes, A, dropout_rate=0.4):
        super().__init__()
        self.layer1 = STGCN_Block(3, 64, A, dropout=dropout_rate)
        self.layer2 = STGCN_Block(64, 64, A, dropout=dropout_rate)
        self.layer3 = STGCN_Block(64, 64, A, dropout=dropout_rate)
        self.layer4 = STGCN_Block(64, 128, A, stride=2, dropout=dropout_rate)
        self.layer5 = STGCN_Block(128, 128, A, dropout=dropout_rate)
        self.layer6 = STGCN_Block(128, 128, A, dropout=dropout_rate)
        self.layer7 = STGCN_Block(128, 256, A, stride=2, dropout=dropout_rate)
        self.layer8 = STGCN_Block(256, 256, A, dropout=dropout_rate)
        self.layer9 = STGCN_Block(256, 256, A, dropout=dropout_rate)
        self.fcn = nn.Conv2d(256, num_classes, 1)

    def forward(self, x):
        for l in [self.layer1,self.layer2,self.layer3,self.layer4,self.layer5,self.layer6,self.layer7,self.layer8,self.layer9]: x = l(x)
        x = F.avg_pool2d(x, x.size()[2:])
        return self.fcn(x).view(x.size(0), -1)

# ==========================================
# 2. TRAINING ENGINE 
# ==========================================
def train_attention_model():
    seed = 42
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    DATASET_DIR = r"C:\Users\User\Documents\Personal Akams\Thesis\Final_Thesis_Dataset_1\Fold_1"
    TRAIN_DIR = os.path.join(DATASET_DIR, "train")
    VAL_DIR = os.path.join(DATASET_DIR, "val")
    
    # 🟢 Descriptive Folder Name for the Ablation
    MODELS_DIR = "Ablation_9Layer_SE_Noise0.005_NoLabelSmoothing"
    os.makedirs(MODELS_DIR, exist_ok=True)
    
    EPOCHS = 50
    BATCH_SIZE = 32
    LEARNING_RATE = 0.001
    DROPOUT = 0.4
    WEIGHT_DECAY = 1e-4
    NOISE_LEVEL = 0.005  # 🟢 SET TO 0.005
    
    # 🟢 THE GPU SCANNER FIX
    target_idx = 0
    print("\n🔍 Scanning available DirectML GPUs...")
    for i in range(torch_directml.device_count()):
        gpu_name = torch_directml.device_name(i)
        print(f"  Found Device {i}: {gpu_name}")
        if "7900" in gpu_name or "GRE" in gpu_name or "RX" in gpu_name: 
            target_idx = i

    dml = torch_directml.device(target_idx)
    print(f"\n🚀 EXPERIMENT STARTED. Locked onto: {torch_directml.device_name(target_idx)}")
    print(f"🔬 Configuration: 9-Layer, SE Attention, Drop 0.4, Noise 0.005, Label Smoothing NONE")
    print(f"🔒 Random Seed locked to {seed} for strict reproducibility.")

    all_classes = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
    class_to_id = {cls_name: idx for idx, cls_name in enumerate(all_classes)}
    id_to_class = {idx: cls_name for cls_name, idx in class_to_id.items()} 
    num_classes = len(all_classes)
    
    homophone_pairs = [
        ("0_Lefthand", "O_Lefthand"), ("0_Righthand", "O_Righthand"),
        ("2_Lefthand", "V_Lefthand"), ("2_Righthand", "V_Righthand")
    ]
    
    allowed_confusions = []
    for a, b in homophone_pairs:
        if a in class_to_id and b in class_to_id:
            allowed_confusions.append(set([class_to_id[a], class_to_id[b]]))

    base_allowed_confusions = []
    for a, b in homophone_pairs:
        base_a = a.split('_')[0] if '_' in a else a
        base_b = b.split('_')[0] if '_' in b else b
        pair = set([base_a, base_b])
        if pair not in base_allowed_confusions:
            base_allowed_confusions.append(pair)

    train_data = CleanBanglaDataset(TRAIN_DIR, class_to_id)
    val_data = CleanBanglaDataset(VAL_DIR, class_to_id)

    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=False)
    val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

    graph = Graph()
    model = BanglaSignSTGCN(num_classes, graph.A, dropout_rate=DROPOUT).to(dml)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, foreach=False)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    
    # 🟢 LABEL SMOOTHING REMOVED
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0

    history_train = []
    history_strict = []
    history_relaxed = []
    history_base = []

    for epoch in range(EPOCHS):
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        
        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs = inputs.to(dml, non_blocking=True)
            labels = labels.to(dml, non_blocking=True)
            
            # 🟢 Noise Added: 0.005
            if NOISE_LEVEL > 0.0:
                noise = torch.randn(*inputs.shape, device=dml) * NOISE_LEVEL
                inputs = inputs + noise
                
            optimizer.zero_grad(set_to_none=True)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * inputs.size(0)
            _, pred = outputs.max(1)
            train_total += labels.size(0)
            train_correct += pred.eq(labels).sum().item()

            if epoch == 0 and (batch_idx + 1) % 100 == 0:
                print(f"   ⏳ Epoch [{epoch+1}/{EPOCHS}] - Processing Batch {batch_idx+1}/{len(train_loader)}...")

        epoch_train_acc = 100. * train_correct / train_total

        model.eval()
        val_loss, strict_correct, relaxed_correct, base_sign_correct, val_total = 0.0, 0, 0, 0, 0
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(dml, non_blocking=True)
                labels = labels.to(dml, non_blocking=True)
                
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                val_total += labels.size(0)
                
                _, top1_preds = outputs.max(1) 
                
                for i in range(labels.size(0)):
                    true_id = labels[i].item()
                    top1_id = top1_preds[i].item()
                    
                    true_name = id_to_class[true_id]
                    pred_name = id_to_class[top1_id]
                    
                    if top1_id == true_id:
                        strict_correct += 1
                        relaxed_correct += 1
                    else:
                        prediction_pair = set([true_id, top1_id])
                        if prediction_pair in allowed_confusions:
                            relaxed_correct += 1
                            
                    true_base = true_name.split('_')[0] if '_' in true_name else true_name
                    pred_base = pred_name.split('_')[0] if '_' in pred_name else pred_name
                    
                    if true_base == pred_base or set([true_base, pred_base]) in base_allowed_confusions:
                        base_sign_correct += 1
                
        strict_acc = 100. * strict_correct / val_total
        relaxed_acc = 100. * relaxed_correct / val_total
        base_sign_acc = 100. * base_sign_correct / val_total 

        history_train.append(epoch_train_acc)
        history_strict.append(strict_acc)
        history_relaxed.append(relaxed_acc)
        history_base.append(base_sign_acc)

        print(f"Epoch [{epoch+1:02d}/{EPOCHS}] | Train: {epoch_train_acc:.2f}% | "
              f"Val Strict: {strict_acc:.2f}% | Relaxed: {relaxed_acc:.2f}% -> Base Sign: {base_sign_acc:.2f}%")

        if relaxed_acc > best_val_acc: 
            best_val_acc = relaxed_acc
            save_path = os.path.join(MODELS_DIR, "9layer_SE_noise_noLS.pth")
            torch.save(model.state_dict(), save_path)
            
        scheduler.step()

    print("\n" + "="*60)
    print(f"🎉 Training Complete! Best Relaxed Val Accuracy: {best_val_acc:.2f}%")
    print("="*60)

    print(f"⏳ Generating Detailed Accuracy Graph in {MODELS_DIR}...")
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, EPOCHS + 1), history_train, label='Train Accuracy', color='blue')
    plt.plot(range(1, EPOCHS + 1), history_strict, label='Simple Val Acc', color='red')
    plt.plot(range(1, EPOCHS + 1), history_relaxed, label='Val Relaxed', color='green')
    plt.plot(range(1, EPOCHS + 1), history_base, label='Val Base Sign', color='orange', linestyle='--')
    
    plt.title('9-Layer SE ST-GCN Accuracy (No Label Smoothing)')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True)
    
    graph_path_detailed = os.path.join(MODELS_DIR, "accuracy_graph_detailed.png")
    plt.savefig(graph_path_detailed)
    plt.close()
    print(f"✅ Saved Detailed Graph to {graph_path_detailed}")

    print(f"⏳ Generating Traditional Accuracy Graph in {MODELS_DIR}...")
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, EPOCHS + 1), history_train, label='Train Accuracy', color='blue')
    plt.plot(range(1, EPOCHS + 1), history_strict, label='Val Accuracy', color='red') 
    
    plt.title('9-Layer SE ST-GCN Accuracy (Train vs Validation)')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True)
    
    graph_path_traditional = os.path.join(MODELS_DIR, "accuracy_graph_traditional.png")
    plt.savefig(graph_path_traditional)
    plt.close()
    print(f"✅ Saved Traditional Graph to {graph_path_traditional}")

if __name__ == "__main__":
    import multiprocessing
    multiprocessing.freeze_support()
    train_attention_model()


🔍 Scanning available DirectML GPUs...
  Found Device 0: AMD Radeon(TM) Graphics 
  Found Device 1: AMD Radeon RX 7900 GRE 

🚀 EXPERIMENT STARTED. Locked onto: AMD Radeon RX 7900 GRE 
🔬 Configuration: 9-Layer, SE Attention, Drop 0.4, Noise 0.005, Label Smoothing NONE
🔒 Random Seed locked to 42 for strict reproducibility.
📂 Booting RAM-Cache Loader for train...
✅ Successfully cached 34200 PyTorch Tensors in Memory!
📂 Booting RAM-Cache Loader for val...
✅ Successfully cached 8681 PyTorch Tensors in Memory!
   ⏳ Epoch [1/50] - Processing Batch 100/1069...
   ⏳ Epoch [1/50] - Processing Batch 200/1069...
   ⏳ Epoch [1/50] - Processing Batch 300/1069...
   ⏳ Epoch [1/50] - Processing Batch 400/1069...
   ⏳ Epoch [1/50] - Processing Batch 500/1069...
   ⏳ Epoch [1/50] - Processing Batch 600/1069...
   ⏳ Epoch [1/50] - Processing Batch 700/1069...
   ⏳ Epoch [1/50] - Processing Batch 800/1069...
   ⏳ Epoch [1/50] - Processing Batch 900/1069...
   ⏳ Epoch [1/50] - Processing Batch 1000/1069...

Depth 9	Temporal Window  9	Act Function ReLU	Attention Mechanism Channel Attention (SE)	
Start Learning Rate 0.001	LR Scheduling Cosine Annealing	
Dropout Rate 0.4	Noise Added 0.005	Label Smoothing 0.2

In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torch_directml
import warnings
import matplotlib.pyplot as plt

# 🟢 Silence the harmless DirectML warning
warnings.filterwarnings("ignore", message=".*aten::lerp.Scalar_out.*")

# ==========================================
# 1. CORE DATA STRUCTURES & MODEL
# ==========================================
class Graph:
    def __init__(self):
        self.num_node = 68  
        self.edges = self._get_edges()
        self.A = self._get_adjacency_matrix()

    def _get_edges(self):
        pose_edges = [
            (0,1), (1,2), (2,3), (3,7),
            (0,4), (4,5), (5,6), (6,8),
            (9,10),
            (11,12), (11,23), (12,24), (23,24),
            (11,13), (13,15),
            (12,14), (14,16),
            (15,17), (15,19), (15,21),
            (16,18), (16,20), (16,22)
        ]
        face_edges = [(0, 25)]
        hand_links = [(0,1), (1,2), (2,3), (3,4),
                      (0,5), (5,6), (6,7), (7,8),
                      (5,9), (9,10), (10,11), (11,12),
                      (9,13), (13,14), (14,15), (15,16),
                      (13,17), (0,17), (17,18), (18,19), (19,20)]
        left_hand_edges = [(s + 26, e + 26) for s, e in hand_links]
        right_hand_edges = [(s + 47, e + 47) for s, e in hand_links]
        connection_edges = [(15, 26), (16, 47)]
        
        return pose_edges + face_edges + left_hand_edges + right_hand_edges + connection_edges

    def _get_adjacency_matrix(self):
        A = np.zeros((self.num_node, self.num_node))
        for i, j in self.edges:
            A[i, j] = 1; A[j, i] = 1
        return torch.tensor(A, dtype=torch.float32)

class CleanBanglaDataset(Dataset):
    def __init__(self, split_dir, class_to_id):
        self.samples = []
        
        print(f"📂 Booting RAM-Cache Loader for {os.path.basename(split_dir)}...")
        
        for class_name in os.listdir(split_dir):
            class_path = os.path.join(split_dir, class_name)
            if not os.path.isdir(class_path): continue
            
            if class_name in class_to_id:
                cid = class_to_id[class_name]
                for f_name in os.listdir(class_path):
                    if f_name.endswith('.npy'):
                        f_path = os.path.join(class_path, f_name)
                        
                        raw_data = np.load(f_path).reshape(90, 68, 3) 
                        raw_data = raw_data - np.mean(raw_data, axis=1, keepdims=True) 
                        raw_data = raw_data.transpose(2, 0, 1)
                        
                        # ⚡ Pre-Tensorize data
                        tensor_data = torch.tensor(raw_data, dtype=torch.float32)
                        tensor_label = torch.tensor(cid, dtype=torch.long)
                        self.samples.append((tensor_data, tensor_label))
                        
        print(f"✅ Successfully cached {len(self.samples)} PyTorch Tensors in Memory!")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx): return self.samples[idx]

class SpatialGraphConv(nn.Module):
    def __init__(self, in_c, out_c, A):
        super().__init__()
        self.register_buffer('A', A)
        self.conv = nn.Conv2d(in_c, out_c, 1)
    def forward(self, x):
        x = torch.einsum('nctv,vw->nctw', (x, self.A))
        return self.conv(x)

# 🟢 Channel Attention Module (Squeeze-and-Excitation)
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=4):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction, in_channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class STGCN_Block(nn.Module):
    def __init__(self, in_c, out_c, A, stride=1, dropout=0.4):
        super().__init__()
        self.sgcn = SpatialGraphConv(in_c, out_c, A)
        self.tgcn = nn.Sequential(
            nn.BatchNorm2d(out_c), 
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, (9, 1), (stride, 1), (4, 0)),
            nn.BatchNorm2d(out_c), 
            nn.Dropout(dropout)
        )
        # 🟢 Attention layer applied after temporal convolution
        self.attention = ChannelAttention(out_c)
        self.res = nn.Sequential(nn.Conv2d(in_c, out_c, 1, (stride, 1)), nn.BatchNorm2d(out_c)) if in_c != out_c or stride != 1 else nn.Identity()
        
    def forward(self, x): 
        # Pass through SGCN -> TGCN -> Attention -> Residual
        sgcn_out = self.sgcn(x)
        tgcn_out = self.tgcn(sgcn_out)
        attended_out = self.attention(tgcn_out)
        return F.relu(attended_out + self.res(x)) 

# 🔴 STANDARD 9-LAYER ARCHITECTURE
class BanglaSignSTGCN(nn.Module):
    def __init__(self, num_classes, A, dropout_rate=0.4):
        super().__init__()
        self.layer1 = STGCN_Block(3, 64, A, dropout=dropout_rate)
        self.layer2 = STGCN_Block(64, 64, A, dropout=dropout_rate)
        self.layer3 = STGCN_Block(64, 64, A, dropout=dropout_rate)
        self.layer4 = STGCN_Block(64, 128, A, stride=2, dropout=dropout_rate)
        self.layer5 = STGCN_Block(128, 128, A, dropout=dropout_rate)
        self.layer6 = STGCN_Block(128, 128, A, dropout=dropout_rate)
        self.layer7 = STGCN_Block(128, 256, A, stride=2, dropout=dropout_rate)
        self.layer8 = STGCN_Block(256, 256, A, dropout=dropout_rate)
        self.layer9 = STGCN_Block(256, 256, A, dropout=dropout_rate)
        self.fcn = nn.Conv2d(256, num_classes, 1)

    def forward(self, x):
        for l in [self.layer1,self.layer2,self.layer3,self.layer4,self.layer5,self.layer6,self.layer7,self.layer8,self.layer9]: x = l(x)
        x = F.avg_pool2d(x, x.size()[2:])
        return self.fcn(x).view(x.size(0), -1)

# ==========================================
# 2. TRAINING ENGINE 
# ==========================================
def train_attention_model():
    seed = 42
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    DATASET_DIR = r"C:\Users\User\Documents\Personal Akams\Thesis\Final_Thesis_Dataset_1\Fold_1"
    TRAIN_DIR = os.path.join(DATASET_DIR, "train")
    VAL_DIR = os.path.join(DATASET_DIR, "val")
    
    # 🟢 Descriptive Folder Name for the Ablation
    MODELS_DIR = "Ablation_9Layer_SE_Noise0.005_LS0.2"
    os.makedirs(MODELS_DIR, exist_ok=True)
    
    EPOCHS = 50
    BATCH_SIZE = 32
    LEARNING_RATE = 0.001
    DROPOUT = 0.4
    WEIGHT_DECAY = 1e-4
    NOISE_LEVEL = 0.005  
    
    # 🟢 THE GPU SCANNER FIX
    target_idx = 0
    print("\n🔍 Scanning available DirectML GPUs...")
    for i in range(torch_directml.device_count()):
        gpu_name = torch_directml.device_name(i)
        print(f"  Found Device {i}: {gpu_name}")
        if "7900" in gpu_name or "GRE" in gpu_name or "RX" in gpu_name: 
            target_idx = i

    dml = torch_directml.device(target_idx)
    print(f"\n🚀 EXPERIMENT STARTED. Locked onto: {torch_directml.device_name(target_idx)}")
    print(f"🔬 Configuration: 9-Layer, SE Attention, Drop 0.4, Noise 0.005, Label Smoothing 0.2")
    print(f"🔒 Random Seed locked to {seed} for strict reproducibility.")

    all_classes = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
    class_to_id = {cls_name: idx for idx, cls_name in enumerate(all_classes)}
    id_to_class = {idx: cls_name for cls_name, idx in class_to_id.items()} 
    num_classes = len(all_classes)
    
    homophone_pairs = [
        ("0_Lefthand", "O_Lefthand"), ("0_Righthand", "O_Righthand"),
        ("2_Lefthand", "V_Lefthand"), ("2_Righthand", "V_Righthand")
    ]
    
    allowed_confusions = []
    for a, b in homophone_pairs:
        if a in class_to_id and b in class_to_id:
            allowed_confusions.append(set([class_to_id[a], class_to_id[b]]))

    base_allowed_confusions = []
    for a, b in homophone_pairs:
        base_a = a.split('_')[0] if '_' in a else a
        base_b = b.split('_')[0] if '_' in b else b
        pair = set([base_a, base_b])
        if pair not in base_allowed_confusions:
            base_allowed_confusions.append(pair)

    train_data = CleanBanglaDataset(TRAIN_DIR, class_to_id)
    val_data = CleanBanglaDataset(VAL_DIR, class_to_id)

    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=False)
    val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

    graph = Graph()
    model = BanglaSignSTGCN(num_classes, graph.A, dropout_rate=DROPOUT).to(dml)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, foreach=False)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    
    # 🟢 LABEL SMOOTHING INCREASED TO 0.2
    criterion = nn.CrossEntropyLoss(label_smoothing=0.2)

    best_val_acc = 0.0

    history_train = []
    history_strict = []
    history_relaxed = []
    history_base = []

    for epoch in range(EPOCHS):
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        
        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs = inputs.to(dml, non_blocking=True)
            labels = labels.to(dml, non_blocking=True)
            
            # 🟢 Logic remains, but skips execution since NOISE_LEVEL is 0.0
            if NOISE_LEVEL > 0.0:
                noise = torch.randn(*inputs.shape, device=dml) * NOISE_LEVEL
                inputs = inputs + noise
                
            optimizer.zero_grad(set_to_none=True)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * inputs.size(0)
            _, pred = outputs.max(1)
            train_total += labels.size(0)
            train_correct += pred.eq(labels).sum().item()

            if epoch == 0 and (batch_idx + 1) % 100 == 0:
                print(f"   ⏳ Epoch [{epoch+1}/{EPOCHS}] - Processing Batch {batch_idx+1}/{len(train_loader)}...")

        epoch_train_acc = 100. * train_correct / train_total

        model.eval()
        val_loss, strict_correct, relaxed_correct, base_sign_correct, val_total = 0.0, 0, 0, 0, 0
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(dml, non_blocking=True)
                labels = labels.to(dml, non_blocking=True)
                
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                val_total += labels.size(0)
                
                _, top1_preds = outputs.max(1) 
                
                for i in range(labels.size(0)):
                    true_id = labels[i].item()
                    top1_id = top1_preds[i].item()
                    
                    true_name = id_to_class[true_id]
                    pred_name = id_to_class[top1_id]
                    
                    if top1_id == true_id:
                        strict_correct += 1
                        relaxed_correct += 1
                    else:
                        prediction_pair = set([true_id, top1_id])
                        if prediction_pair in allowed_confusions:
                            relaxed_correct += 1
                            
                    true_base = true_name.split('_')[0] if '_' in true_name else true_name
                    pred_base = pred_name.split('_')[0] if '_' in pred_name else pred_name
                    
                    if true_base == pred_base or set([true_base, pred_base]) in base_allowed_confusions:
                        base_sign_correct += 1
                
        strict_acc = 100. * strict_correct / val_total
        relaxed_acc = 100. * relaxed_correct / val_total
        base_sign_acc = 100. * base_sign_correct / val_total 

        history_train.append(epoch_train_acc)
        history_strict.append(strict_acc)
        history_relaxed.append(relaxed_acc)
        history_base.append(base_sign_acc)

        print(f"Epoch [{epoch+1:02d}/{EPOCHS}] | Train: {epoch_train_acc:.2f}% | "
              f"Val Strict: {strict_acc:.2f}% | Relaxed: {relaxed_acc:.2f}% -> Base Sign: {base_sign_acc:.2f}%")

        if relaxed_acc > best_val_acc: 
            best_val_acc = relaxed_acc
            save_path = os.path.join(MODELS_DIR, "9layer_SE_nonoise_LS0.2.pth")
            torch.save(model.state_dict(), save_path)
            
        scheduler.step()

    print("\n" + "="*60)
    print(f"🎉 Training Complete! Best Relaxed Val Accuracy: {best_val_acc:.2f}%")
    print("="*60)

    print(f"⏳ Generating Detailed Accuracy Graph in {MODELS_DIR}...")
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, EPOCHS + 1), history_train, label='Train Accuracy', color='blue')
    plt.plot(range(1, EPOCHS + 1), history_strict, label='Simple Val Acc', color='red')
    plt.plot(range(1, EPOCHS + 1), history_relaxed, label='Val Relaxed', color='green')
    plt.plot(range(1, EPOCHS + 1), history_base, label='Val Base Sign', color='orange', linestyle='--')
    
    plt.title('9-Layer SE ST-GCN Accuracy (No Noise, LS=0.2)')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True)
    
    graph_path_detailed = os.path.join(MODELS_DIR, "accuracy_graph_detailed.png")
    plt.savefig(graph_path_detailed)
    plt.close()
    print(f"✅ Saved Detailed Graph to {graph_path_detailed}")

    print(f"⏳ Generating Traditional Accuracy Graph in {MODELS_DIR}...")
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, EPOCHS + 1), history_train, label='Train Accuracy', color='blue')
    plt.plot(range(1, EPOCHS + 1), history_strict, label='Val Accuracy', color='red') 
    
    plt.title('9-Layer SE ST-GCN Accuracy (Train vs Validation)')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True)
    
    graph_path_traditional = os.path.join(MODELS_DIR, "accuracy_graph_traditional.png")
    plt.savefig(graph_path_traditional)
    plt.close()
    print(f"✅ Saved Traditional Graph to {graph_path_traditional}")

if __name__ == "__main__":
    import multiprocessing
    multiprocessing.freeze_support()
    train_attention_model()


🔍 Scanning available DirectML GPUs...
  Found Device 0: AMD Radeon(TM) Graphics 
  Found Device 1: AMD Radeon RX 7900 GRE 

🚀 EXPERIMENT STARTED. Locked onto: AMD Radeon RX 7900 GRE 
🔬 Configuration: 9-Layer, SE Attention, Drop 0.4, Noise 0.005, Label Smoothing 0.2
🔒 Random Seed locked to 42 for strict reproducibility.
📂 Booting RAM-Cache Loader for train...
✅ Successfully cached 34200 PyTorch Tensors in Memory!
📂 Booting RAM-Cache Loader for val...
✅ Successfully cached 8681 PyTorch Tensors in Memory!
   ⏳ Epoch [1/50] - Processing Batch 100/1069...
   ⏳ Epoch [1/50] - Processing Batch 200/1069...
   ⏳ Epoch [1/50] - Processing Batch 300/1069...
   ⏳ Epoch [1/50] - Processing Batch 400/1069...
   ⏳ Epoch [1/50] - Processing Batch 500/1069...
   ⏳ Epoch [1/50] - Processing Batch 600/1069...
   ⏳ Epoch [1/50] - Processing Batch 700/1069...
   ⏳ Epoch [1/50] - Processing Batch 800/1069...
   ⏳ Epoch [1/50] - Processing Batch 900/1069...
   ⏳ Epoch [1/50] - Processing Batch 1000/1069...


Depth 9	Temporal Window  9	Act Function ReLU	Attention Mechanism Spatial Attention (all Layer)
Start Learning Rate 0.001	LR Scheduling Cosine Annealing	
Dropout Rate 0.4	Noise Added 0.005	Label Smoothing 0.1

In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torch_directml
import warnings
import matplotlib.pyplot as plt
import gc

# 🟢 Silence the harmless DirectML warning
warnings.filterwarnings("ignore", message=".*aten::lerp.Scalar_out.*")

# ==========================================
# 1. CORE DATA STRUCTURES & MODEL
# ==========================================
class Graph:
    def __init__(self):
        self.num_node = 68  
        self.edges = self._get_edges()
        self.A = self._get_adjacency_matrix()

    def _get_edges(self):
        pose_edges = [
            (0,1), (1,2), (2,3), (3,7),
            (0,4), (4,5), (5,6), (6,8),
            (9,10),
            (11,12), (11,23), (12,24), (23,24),
            (11,13), (13,15),
            (12,14), (14,16),
            (15,17), (15,19), (15,21),
            (16,18), (16,20), (16,22)
        ]
        face_edges = [(0, 25)]
        hand_links = [(0,1), (1,2), (2,3), (3,4),
                      (0,5), (5,6), (6,7), (7,8),
                      (5,9), (9,10), (10,11), (11,12),
                      (9,13), (13,14), (14,15), (15,16),
                      (13,17), (0,17), (17,18), (18,19), (19,20)]
        left_hand_edges = [(s + 26, e + 26) for s, e in hand_links]
        right_hand_edges = [(s + 47, e + 47) for s, e in hand_links]
        connection_edges = [(15, 26), (16, 47)]
        
        return pose_edges + face_edges + left_hand_edges + right_hand_edges + connection_edges

    def _get_adjacency_matrix(self):
        A = np.zeros((self.num_node, self.num_node))
        for i, j in self.edges:
            A[i, j] = 1; A[j, i] = 1
        return torch.tensor(A, dtype=torch.float32)

class CleanBanglaDataset(Dataset):
    def __init__(self, split_dir, class_to_id):
        self.samples = []
        
        for class_name in os.listdir(split_dir):
            class_path = os.path.join(split_dir, class_name)
            if not os.path.isdir(class_path): continue
            
            if class_name in class_to_id:
                cid = class_to_id[class_name]
                for f_name in os.listdir(class_path):
                    if f_name.endswith('.npy'):
                        f_path = os.path.join(class_path, f_name)
                        
                        raw_data = np.load(f_path).reshape(90, 68, 3) 
                        raw_data = raw_data - np.mean(raw_data, axis=1, keepdims=True) 
                        raw_data = raw_data.transpose(2, 0, 1)
                        
                        tensor_data = torch.tensor(raw_data, dtype=torch.float32)
                        tensor_label = torch.tensor(cid, dtype=torch.long)
                        self.samples.append((tensor_data, tensor_label))
                        
    def __len__(self): return len(self.samples)

    def __getitem__(self, idx): return self.samples[idx]

class SpatialGraphConv(nn.Module):
    def __init__(self, in_c, out_c, A):
        super().__init__()
        self.register_buffer('A', A)
        self.conv = nn.Conv2d(in_c, out_c, 1)
    def forward(self, x):
        x = torch.einsum('nctv,vw->nctw', (x, self.A))
        return self.conv(x)

# 🟢 Spatial Self-Attention Module
class SpatialSelfAttention(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        reduced_channels = max(1, in_channels // 8)
        self.query_conv = nn.Conv2d(in_channels, reduced_channels, 1)
        self.key_conv = nn.Conv2d(in_channels, reduced_channels, 1)
        self.value_conv = nn.Conv2d(in_channels, in_channels, 1)
        self.gamma = nn.Parameter(torch.zeros(1))
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):
        q = self.query_conv(x)
        k = self.key_conv(x)
        v = self.value_conv(x)
        
        energy = torch.einsum('nctv,nctu->ntvu', q, k)
        attention = self.softmax(energy)
        
        out = torch.einsum('nctu,ntvu->nctv', v, attention)
        return self.gamma * out + x

class STGCN_Block(nn.Module):
    def __init__(self, in_c, out_c, A, stride=1, dropout=0.4):
        super().__init__()
        self.sgcn = SpatialGraphConv(in_c, out_c, A)
        self.tgcn = nn.Sequential(
            nn.BatchNorm2d(out_c), 
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, (9, 1), (stride, 1), (4, 0)),
            nn.BatchNorm2d(out_c), 
            nn.Dropout(dropout)
        )
        # 🟢 Attention layer: SpatialSelfAttention
        self.attention = SpatialSelfAttention(out_c)
        self.res = nn.Sequential(nn.Conv2d(in_c, out_c, 1, (stride, 1)), nn.BatchNorm2d(out_c)) if in_c != out_c or stride != 1 else nn.Identity()
        
    def forward(self, x): 
        sgcn_out = self.sgcn(x)
        tgcn_out = self.tgcn(sgcn_out)
        attended_out = self.attention(tgcn_out)
        return F.relu(attended_out + self.res(x)) 

class BanglaSignSTGCN(nn.Module):
    def __init__(self, num_classes, A, dropout_rate=0.4):
        super().__init__()
        self.layer1 = STGCN_Block(3, 64, A, dropout=dropout_rate)
        self.layer2 = STGCN_Block(64, 64, A, dropout=dropout_rate)
        self.layer3 = STGCN_Block(64, 64, A, dropout=dropout_rate)
        self.layer4 = STGCN_Block(64, 128, A, stride=2, dropout=dropout_rate)
        self.layer5 = STGCN_Block(128, 128, A, dropout=dropout_rate)
        self.layer6 = STGCN_Block(128, 128, A, dropout=dropout_rate)
        self.layer7 = STGCN_Block(128, 256, A, stride=2, dropout=dropout_rate)
        self.layer8 = STGCN_Block(256, 256, A, dropout=dropout_rate)
        self.layer9 = STGCN_Block(256, 256, A, dropout=dropout_rate)
        self.fcn = nn.Conv2d(256, num_classes, 1)

    def forward(self, x):
        for l in [self.layer1,self.layer2,self.layer3,self.layer4,self.layer5,self.layer6,self.layer7,self.layer8,self.layer9]: x = l(x)
        x = F.avg_pool2d(x, x.size()[2:])
        return self.fcn(x).view(x.size(0), -1)


# ==========================================
# 2. AUTO-RECOVERY TRAINING ENGINE
# ==========================================
def train_attention_model():
    seed = 42
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    DATASET_DIR = r"C:\Users\User\Documents\Personal Akams\Thesis\Final_Thesis_Dataset_1\Fold_1"
    TRAIN_DIR = os.path.join(DATASET_DIR, "train")
    VAL_DIR = os.path.join(DATASET_DIR, "val")
    
    MODELS_DIR = "Ablation_9Layer_SpatialAtt_Noise0.005_LS0.1"
    os.makedirs(MODELS_DIR, exist_ok=True)
    
    EPOCHS = 50
    LEARNING_RATE = 0.001
    DROPOUT = 0.4
    WEIGHT_DECAY = 1e-4
    NOISE_LEVEL = 0.005  
    
    # Pre-cache dataset to avoid reloading it on every retry
    print("📂 Booting RAM-Cache Loader...")
    all_classes = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
    class_to_id = {cls_name: idx for idx, cls_name in enumerate(all_classes)}
    id_to_class = {idx: cls_name for cls_name, idx in class_to_id.items()} 
    num_classes = len(all_classes)
    
    homophone_pairs = [
        ("0_Lefthand", "O_Lefthand"), ("0_Righthand", "O_Righthand"),
        ("2_Lefthand", "V_Lefthand"), ("2_Righthand", "V_Righthand")
    ]
    allowed_confusions = [set([class_to_id[a], class_to_id[b]]) for a, b in homophone_pairs if a in class_to_id and b in class_to_id]
    
    base_allowed_confusions = []
    for a, b in homophone_pairs:
        base_a = a.split('_')[0] if '_' in a else a
        base_b = b.split('_')[0] if '_' in b else b
        pair = set([base_a, base_b])
        if pair not in base_allowed_confusions:
            base_allowed_confusions.append(pair)

    train_data = CleanBanglaDataset(TRAIN_DIR, class_to_id)
    val_data = CleanBanglaDataset(VAL_DIR, class_to_id)
    
    # 🟢 AUTO-RECOVERY BATCH SIZING LOGIC
    current_batch_size = 32

    while current_batch_size >= 2:
        try:
            print(f"\n=======================================================")
            print(f"🚀 ATTEMPTING TRAINING WITH BATCH SIZE: {current_batch_size}")
            print(f"=======================================================\n")
            
            target_idx = 0
            for i in range(torch_directml.device_count()):
                gpu_name = torch_directml.device_name(i)
                if "7900" in gpu_name or "GRE" in gpu_name or "RX" in gpu_name: 
                    target_idx = i

            dml = torch_directml.device(target_idx)
            print(f"Locked onto: {torch_directml.device_name(target_idx)}")

            train_loader = DataLoader(train_data, batch_size=current_batch_size, shuffle=True, num_workers=0, pin_memory=False)
            val_loader = DataLoader(val_data, batch_size=current_batch_size, shuffle=False, num_workers=0, pin_memory=False)

            graph = Graph()
            model = BanglaSignSTGCN(num_classes, graph.A, dropout_rate=DROPOUT).to(dml)
            
            optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, foreach=False)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
            criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

            best_val_acc = 0.0
            history_train, history_strict, history_relaxed, history_base = [], [], [], []

            for epoch in range(EPOCHS):
                model.train()
                train_loss, train_correct, train_total = 0.0, 0, 0
                
                for batch_idx, (inputs, labels) in enumerate(train_loader):
                    inputs = inputs.to(dml, non_blocking=True)
                    labels = labels.to(dml, non_blocking=True)
                    
                    if NOISE_LEVEL > 0.0:
                        noise = torch.randn(*inputs.shape, device=dml) * NOISE_LEVEL
                        inputs = inputs + noise
                        
                    optimizer.zero_grad(set_to_none=True)
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    
                    loss.backward()
                    optimizer.step()
                    
                    train_loss += loss.item() * inputs.size(0)
                    _, pred = outputs.max(1)
                    train_total += labels.size(0)
                    train_correct += pred.eq(labels).sum().item()

                    if epoch == 0 and (batch_idx + 1) % 100 == 0:
                        print(f"   ⏳ Epoch [{epoch+1}/{EPOCHS}] - Processing Batch {batch_idx+1}/{len(train_loader)}...")

                epoch_train_acc = 100. * train_correct / train_total

                model.eval()
                val_loss, strict_correct, relaxed_correct, base_sign_correct, val_total = 0.0, 0, 0, 0, 0
                
                with torch.no_grad():
                    for inputs, labels in val_loader:
                        inputs = inputs.to(dml, non_blocking=True)
                        labels = labels.to(dml, non_blocking=True)
                        
                        outputs = model(inputs)
                        loss = criterion(outputs, labels)
                        val_loss += loss.item() * inputs.size(0)
                        val_total += labels.size(0)
                        
                        _, top1_preds = outputs.max(1) 
                        
                        for i in range(labels.size(0)):
                            true_id = labels[i].item()
                            top1_id = top1_preds[i].item()
                            
                            true_name = id_to_class[true_id]
                            pred_name = id_to_class[top1_id]
                            
                            if top1_id == true_id:
                                strict_correct += 1
                                relaxed_correct += 1
                            else:
                                prediction_pair = set([true_id, top1_id])
                                if prediction_pair in allowed_confusions:
                                    relaxed_correct += 1
                                    
                            true_base = true_name.split('_')[0] if '_' in true_name else true_name
                            pred_base = pred_name.split('_')[0] if '_' in pred_name else pred_name
                            
                            if true_base == pred_base or set([true_base, pred_base]) in base_allowed_confusions:
                                base_sign_correct += 1
                        
                strict_acc = 100. * strict_correct / val_total
                relaxed_acc = 100. * relaxed_correct / val_total
                base_sign_acc = 100. * base_sign_correct / val_total 

                history_train.append(epoch_train_acc)
                history_strict.append(strict_acc)
                history_relaxed.append(relaxed_acc)
                history_base.append(base_sign_acc)

                print(f"Epoch [{epoch+1:02d}/{EPOCHS}] | Train: {epoch_train_acc:.2f}% | "
                      f"Val Strict: {strict_acc:.2f}% | Relaxed: {relaxed_acc:.2f}% -> Base Sign: {base_sign_acc:.2f}%")

                if relaxed_acc > best_val_acc: 
                    best_val_acc = relaxed_acc
                    save_path = os.path.join(MODELS_DIR, f"9layer_Spatial_Batch{current_batch_size}.pth")
                    torch.save(model.state_dict(), save_path)
                    
                scheduler.step()
            
            # If the loop completes 50 epochs without crashing, break the while loop!
            print("\n🎉 Training Complete Successfully!")
            break 
            
        except RuntimeError as e:
            error_message = str(e).lower()
            if "gpu will not respond" in error_message or "memory" in error_message:
                print(f"\n🛑 CRASH DETECTED: GPU Timeout/OOM with Batch Size {current_batch_size}.")
                print(f"   Reason: {str(e)}")
                
                # Halve the batch size and clean memory
                current_batch_size = current_batch_size // 2
                print(f"🔄 Reducing Batch Size to {current_batch_size} and restarting...\n")
                
                # Nuke the broken model and variables from RAM to allow a clean restart
                del model
                del optimizer
                del train_loader
                del val_loader
                gc.collect()
            else:
                # If it's a different code error (like a typo), crash normally
                raise e

    # 🟢 Generation of graphs after successful loop break
    print(f"⏳ Generating Detailed Accuracy Graph in {MODELS_DIR}...")
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, EPOCHS + 1), history_train, label='Train Accuracy', color='blue')
    plt.plot(range(1, EPOCHS + 1), history_strict, label='Simple Val Acc', color='red')
    plt.plot(range(1, EPOCHS + 1), history_relaxed, label='Val Relaxed', color='green')
    plt.plot(range(1, EPOCHS + 1), history_base, label='Val Base Sign', color='orange', linestyle='--')
    
    plt.title('9-Layer Spatial Attention Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True)
    
    graph_path_detailed = os.path.join(MODELS_DIR, "accuracy_graph_detailed.png")
    plt.savefig(graph_path_detailed)
    plt.close()

if __name__ == "__main__":
    import multiprocessing
    multiprocessing.freeze_support()
    train_attention_model()

📂 Booting RAM-Cache Loader...

🚀 ATTEMPTING TRAINING WITH BATCH SIZE: 32

Locked onto: AMD Radeon RX 7900 GRE 
   ⏳ Epoch [1/50] - Processing Batch 100/1069...
   ⏳ Epoch [1/50] - Processing Batch 200/1069...
   ⏳ Epoch [1/50] - Processing Batch 300/1069...
   ⏳ Epoch [1/50] - Processing Batch 400/1069...
   ⏳ Epoch [1/50] - Processing Batch 500/1069...
   ⏳ Epoch [1/50] - Processing Batch 600/1069...
   ⏳ Epoch [1/50] - Processing Batch 700/1069...
   ⏳ Epoch [1/50] - Processing Batch 800/1069...
   ⏳ Epoch [1/50] - Processing Batch 900/1069...
   ⏳ Epoch [1/50] - Processing Batch 1000/1069...
Epoch [01/50] | Train: 10.94% | Val Strict: 22.81% | Relaxed: 22.81% -> Base Sign: 24.27%
Epoch [02/50] | Train: 38.02% | Val Strict: 49.54% | Relaxed: 49.64% -> Base Sign: 51.84%
Epoch [03/50] | Train: 56.56% | Val Strict: 65.48% | Relaxed: 65.64% -> Base Sign: 66.66%
Epoch [04/50] | Train: 65.82% | Val Strict: 66.64% | Relaxed: 66.73% -> Base Sign: 67.90%
Epoch [05/50] | Train: 71.63% | Val St

Depth 9	Temporal Window  9	Act Function Swish	Attention Mechanism Channel Attention (SE)	
Start Learning Rate 0.001	LR Scheduling Cosine Annealing	
Dropout Rate 0.4	Noise Added 0.005	Label Smoothing 0.1

In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torch_directml
import warnings
import matplotlib.pyplot as plt

# 🟢 Silence the harmless DirectML warning
warnings.filterwarnings("ignore", message=".*aten::lerp.Scalar_out.*")

# ==========================================
# 1. CORE DATA STRUCTURES & MODEL
# ==========================================
class Graph:
    def __init__(self):
        self.num_node = 68  
        self.edges = self._get_edges()
        self.A = self._get_adjacency_matrix()

    def _get_edges(self):
        pose_edges = [
            (0,1), (1,2), (2,3), (3,7),
            (0,4), (4,5), (5,6), (6,8),
            (9,10),
            (11,12), (11,23), (12,24), (23,24),
            (11,13), (13,15),
            (12,14), (14,16),
            (15,17), (15,19), (15,21),
            (16,18), (16,20), (16,22)
        ]
        face_edges = [(0, 25)]
        hand_links = [(0,1), (1,2), (2,3), (3,4),
                      (0,5), (5,6), (6,7), (7,8),
                      (5,9), (9,10), (10,11), (11,12),
                      (9,13), (13,14), (14,15), (15,16),
                      (13,17), (0,17), (17,18), (18,19), (19,20)]
        left_hand_edges = [(s + 26, e + 26) for s, e in hand_links]
        right_hand_edges = [(s + 47, e + 47) for s, e in hand_links]
        connection_edges = [(15, 26), (16, 47)]
        
        return pose_edges + face_edges + left_hand_edges + right_hand_edges + connection_edges

    def _get_adjacency_matrix(self):
        A = np.zeros((self.num_node, self.num_node))
        for i, j in self.edges:
            A[i, j] = 1; A[j, i] = 1
        return torch.tensor(A, dtype=torch.float32)

class CleanBanglaDataset(Dataset):
    def __init__(self, split_dir, class_to_id):
        self.samples = []
        
        print(f"📂 Booting RAM-Cache Loader for {os.path.basename(split_dir)}...")
        for class_name in os.listdir(split_dir):
            class_path = os.path.join(split_dir, class_name)
            if not os.path.isdir(class_path): continue
            
            if class_name in class_to_id:
                cid = class_to_id[class_name]
                for f_name in os.listdir(class_path):
                    if f_name.endswith('.npy'):
                        f_path = os.path.join(class_path, f_name)
                        
                        raw_data = np.load(f_path).reshape(90, 68, 3) 
                        raw_data = raw_data - np.mean(raw_data, axis=1, keepdims=True) 
                        raw_data = raw_data.transpose(2, 0, 1)
                        
                        tensor_data = torch.tensor(raw_data, dtype=torch.float32)
                        tensor_label = torch.tensor(cid, dtype=torch.long)
                        self.samples.append((tensor_data, tensor_label))
        print(f"✅ Successfully cached {len(self.samples)} PyTorch Tensors in Memory!")
                        
    def __len__(self): return len(self.samples)

    def __getitem__(self, idx): return self.samples[idx]

class SpatialGraphConv(nn.Module):
    def __init__(self, in_c, out_c, A):
        super().__init__()
        self.register_buffer('A', A)
        self.conv = nn.Conv2d(in_c, out_c, 1)
    def forward(self, x):
        x = torch.einsum('nctv,vw->nctw', (x, self.A))
        return self.conv(x)

# 🟢 Channel Attention Module - Updated to Swish (SiLU)
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=4):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.SiLU(),  # 🟢 Swish Activation
            nn.Linear(in_channels // reduction, in_channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class STGCN_Block(nn.Module):
    def __init__(self, in_c, out_c, A, stride=1, dropout=0.4):
        super().__init__()
        self.sgcn = SpatialGraphConv(in_c, out_c, A)
        self.tgcn = nn.Sequential(
            nn.BatchNorm2d(out_c), 
            nn.SiLU(),  # 🟢 Swish Activation
            nn.Conv2d(out_c, out_c, (9, 1), (stride, 1), (4, 0)),
            nn.BatchNorm2d(out_c), 
            nn.Dropout(dropout)
        )
        self.attention = ChannelAttention(out_c)
        self.res = nn.Sequential(nn.Conv2d(in_c, out_c, 1, (stride, 1)), nn.BatchNorm2d(out_c)) if in_c != out_c or stride != 1 else nn.Identity()
        
    def forward(self, x): 
        sgcn_out = self.sgcn(x)
        tgcn_out = self.tgcn(sgcn_out)
        attended_out = self.attention(tgcn_out)
        # 🟢 Forward pass output updated to Swish
        return F.silu(attended_out + self.res(x)) 

class BanglaSignSTGCN(nn.Module):
    def __init__(self, num_classes, A, dropout_rate=0.4):
        super().__init__()
        self.layer1 = STGCN_Block(3, 64, A, dropout=dropout_rate)
        self.layer2 = STGCN_Block(64, 64, A, dropout=dropout_rate)
        self.layer3 = STGCN_Block(64, 64, A, dropout=dropout_rate)
        self.layer4 = STGCN_Block(64, 128, A, stride=2, dropout=dropout_rate)
        self.layer5 = STGCN_Block(128, 128, A, dropout=dropout_rate)
        self.layer6 = STGCN_Block(128, 128, A, dropout=dropout_rate)
        self.layer7 = STGCN_Block(128, 256, A, stride=2, dropout=dropout_rate)
        self.layer8 = STGCN_Block(256, 256, A, dropout=dropout_rate)
        self.layer9 = STGCN_Block(256, 256, A, dropout=dropout_rate)
        self.fcn = nn.Conv2d(256, num_classes, 1)

    def forward(self, x):
        for l in [self.layer1,self.layer2,self.layer3,self.layer4,self.layer5,self.layer6,self.layer7,self.layer8,self.layer9]: x = l(x)
        x = F.avg_pool2d(x, x.size()[2:])
        return self.fcn(x).view(x.size(0), -1)

# ==========================================
# 2. TRAINING ENGINE
# ==========================================
def train_attention_model():
    seed = 42
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    DATASET_DIR = r"C:\Users\User\Documents\Personal Akams\Thesis\Final_Thesis_Dataset_1\Fold_1"
    TRAIN_DIR = os.path.join(DATASET_DIR, "train")
    VAL_DIR = os.path.join(DATASET_DIR, "val")
    
    MODELS_DIR = "Ablation_9Layer_SE_Swish_Noise0.005_LS0.1"
    os.makedirs(MODELS_DIR, exist_ok=True)
    
    EPOCHS = 50
    BATCH_SIZE = 32
    LEARNING_RATE = 0.001
    DROPOUT = 0.4
    WEIGHT_DECAY = 1e-4
    NOISE_LEVEL = 0.005  
    
    target_idx = 0
    print("\n🔍 Scanning available DirectML GPUs...")
    for i in range(torch_directml.device_count()):
        gpu_name = torch_directml.device_name(i)
        print(f"  Found Device {i}: {gpu_name}")
        if "7900" in gpu_name or "GRE" in gpu_name or "RX" in gpu_name: 
            target_idx = i

    dml = torch_directml.device(target_idx)
    print(f"\n🚀 EXPERIMENT STARTED. Locked onto: {torch_directml.device_name(target_idx)}")
    print(f"🔬 Configuration: 9-Layer, SE Attention, SWISH, Drop 0.4, Noise 0.005, LS 0.1")
    print(f"🔒 Random Seed locked to {seed} for strict reproducibility.")

    all_classes = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
    class_to_id = {cls_name: idx for idx, cls_name in enumerate(all_classes)}
    id_to_class = {idx: cls_name for cls_name, idx in class_to_id.items()} 
    num_classes = len(all_classes)
    
    homophone_pairs = [
        ("0_Lefthand", "O_Lefthand"), ("0_Righthand", "O_Righthand"),
        ("2_Lefthand", "V_Lefthand"), ("2_Righthand", "V_Righthand")
    ]
    
    allowed_confusions = []
    for a, b in homophone_pairs:
        if a in class_to_id and b in class_to_id:
            allowed_confusions.append(set([class_to_id[a], class_to_id[b]]))

    base_allowed_confusions = []
    for a, b in homophone_pairs:
        base_a = a.split('_')[0] if '_' in a else a
        base_b = b.split('_')[0] if '_' in b else b
        pair = set([base_a, base_b])
        if pair not in base_allowed_confusions:
            base_allowed_confusions.append(pair)

    train_data = CleanBanglaDataset(TRAIN_DIR, class_to_id)
    val_data = CleanBanglaDataset(VAL_DIR, class_to_id)

    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=False)
    val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

    graph = Graph()
    model = BanglaSignSTGCN(num_classes, graph.A, dropout_rate=DROPOUT).to(dml)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, foreach=False)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    
    # 🟢 Label Smoothing 0.1
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    best_val_acc = 0.0

    history_train = []
    history_strict = []
    history_relaxed = []
    history_base = []

    for epoch in range(EPOCHS):
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        
        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs = inputs.to(dml, non_blocking=True)
            labels = labels.to(dml, non_blocking=True)
            
            if NOISE_LEVEL > 0.0:
                noise = torch.randn(*inputs.shape, device=dml) * NOISE_LEVEL
                inputs = inputs + noise
                
            optimizer.zero_grad(set_to_none=True)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * inputs.size(0)
            _, pred = outputs.max(1)
            train_total += labels.size(0)
            train_correct += pred.eq(labels).sum().item()

            if epoch == 0 and (batch_idx + 1) % 100 == 0:
                print(f"   ⏳ Epoch [{epoch+1}/{EPOCHS}] - Processing Batch {batch_idx+1}/{len(train_loader)}...")

        epoch_train_acc = 100. * train_correct / train_total

        model.eval()
        val_loss, strict_correct, relaxed_correct, base_sign_correct, val_total = 0.0, 0, 0, 0, 0
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(dml, non_blocking=True)
                labels = labels.to(dml, non_blocking=True)
                
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                val_total += labels.size(0)
                
                _, top1_preds = outputs.max(1) 
                
                for i in range(labels.size(0)):
                    true_id = labels[i].item()
                    top1_id = top1_preds[i].item()
                    
                    true_name = id_to_class[true_id]
                    pred_name = id_to_class[top1_id]
                    
                    if top1_id == true_id:
                        strict_correct += 1
                        relaxed_correct += 1
                    else:
                        prediction_pair = set([true_id, top1_id])
                        if prediction_pair in allowed_confusions:
                            relaxed_correct += 1
                            
                    true_base = true_name.split('_')[0] if '_' in true_name else true_name
                    pred_base = pred_name.split('_')[0] if '_' in pred_name else pred_name
                    
                    if true_base == pred_base or set([true_base, pred_base]) in base_allowed_confusions:
                        base_sign_correct += 1
                
        strict_acc = 100. * strict_correct / val_total
        relaxed_acc = 100. * relaxed_correct / val_total
        base_sign_acc = 100. * base_sign_correct / val_total 

        history_train.append(epoch_train_acc)
        history_strict.append(strict_acc)
        history_relaxed.append(relaxed_acc)
        history_base.append(base_sign_acc)

        print(f"Epoch [{epoch+1:02d}/{EPOCHS}] | Train: {epoch_train_acc:.2f}% | "
              f"Val Strict: {strict_acc:.2f}% | Relaxed: {relaxed_acc:.2f}% -> Base Sign: {base_sign_acc:.2f}%")

        if relaxed_acc > best_val_acc: 
            best_val_acc = relaxed_acc
            save_path = os.path.join(MODELS_DIR, "9layer_SE_Swish_Noise0.005_LS0.1.pth")
            torch.save(model.state_dict(), save_path)
            
        scheduler.step()

    print("\n" + "="*60)
    print(f"🎉 Training Complete! Best Relaxed Val Accuracy: {best_val_acc:.2f}%")
    print("="*60)

    print(f"⏳ Generating Detailed Accuracy Graph in {MODELS_DIR}...")
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, EPOCHS + 1), history_train, label='Train Accuracy', color='blue')
    plt.plot(range(1, EPOCHS + 1), history_strict, label='Simple Val Acc', color='red')
    plt.plot(range(1, EPOCHS + 1), history_relaxed, label='Val Relaxed', color='green')
    plt.plot(range(1, EPOCHS + 1), history_base, label='Val Base Sign', color='orange', linestyle='--')
    
    plt.title('9-Layer SE (Swish) Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True)
    
    graph_path_detailed = os.path.join(MODELS_DIR, "accuracy_graph_detailed.png")
    plt.savefig(graph_path_detailed)
    plt.close()
    print(f"✅ Saved Detailed Graph to {graph_path_detailed}")

    print(f"⏳ Generating Traditional Accuracy Graph in {MODELS_DIR}...")
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, EPOCHS + 1), history_train, label='Train Accuracy', color='blue')
    plt.plot(range(1, EPOCHS + 1), history_strict, label='Val Accuracy', color='red') 
    
    plt.title('9-Layer SE (Swish) Accuracy (Train vs Validation)')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True)
    
    graph_path_traditional = os.path.join(MODELS_DIR, "accuracy_graph_traditional.png")
    plt.savefig(graph_path_traditional)
    plt.close()
    print(f"✅ Saved Traditional Graph to {graph_path_traditional}")

if __name__ == "__main__":
    import multiprocessing
    multiprocessing.freeze_support()
    train_attention_model()


🔍 Scanning available DirectML GPUs...
  Found Device 0: AMD Radeon(TM) Graphics 
  Found Device 1: AMD Radeon RX 7900 GRE 

🚀 EXPERIMENT STARTED. Locked onto: AMD Radeon RX 7900 GRE 
🔬 Configuration: 9-Layer, SE Attention, SWISH, Drop 0.4, Noise 0.005, LS 0.1
🔒 Random Seed locked to 42 for strict reproducibility.
📂 Booting RAM-Cache Loader for train...
✅ Successfully cached 34200 PyTorch Tensors in Memory!
📂 Booting RAM-Cache Loader for val...
✅ Successfully cached 8681 PyTorch Tensors in Memory!
   ⏳ Epoch [1/50] - Processing Batch 100/1069...
   ⏳ Epoch [1/50] - Processing Batch 200/1069...
   ⏳ Epoch [1/50] - Processing Batch 300/1069...
   ⏳ Epoch [1/50] - Processing Batch 400/1069...
   ⏳ Epoch [1/50] - Processing Batch 500/1069...
   ⏳ Epoch [1/50] - Processing Batch 600/1069...
   ⏳ Epoch [1/50] - Processing Batch 700/1069...
   ⏳ Epoch [1/50] - Processing Batch 800/1069...
   ⏳ Epoch [1/50] - Processing Batch 900/1069...
   ⏳ Epoch [1/50] - Processing Batch 1000/1069...
Epoch 

Depth 9	Temporal Window  9	Act Function Gelu	Attention Mechanism Channel Attention (SE)	
Start Learning Rate 0.001	LR Scheduling Cosine Annealing	
Dropout Rate 0.4	Noise Added 0.005	Label Smoothing 0.1

In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torch_directml
import warnings
import matplotlib.pyplot as plt

# 🟢 Silence the harmless DirectML warning
warnings.filterwarnings("ignore", message=".*aten::lerp.Scalar_out.*")

# ==========================================
# 1. CORE DATA STRUCTURES & MODEL
# ==========================================
class Graph:
    def __init__(self):
        self.num_node = 68  
        self.edges = self._get_edges()
        self.A = self._get_adjacency_matrix()

    def _get_edges(self):
        pose_edges = [
            (0,1), (1,2), (2,3), (3,7),
            (0,4), (4,5), (5,6), (6,8),
            (9,10),
            (11,12), (11,23), (12,24), (23,24),
            (11,13), (13,15),
            (12,14), (14,16),
            (15,17), (15,19), (15,21),
            (16,18), (16,20), (16,22)
        ]
        face_edges = [(0, 25)]
        hand_links = [(0,1), (1,2), (2,3), (3,4),
                      (0,5), (5,6), (6,7), (7,8),
                      (5,9), (9,10), (10,11), (11,12),
                      (9,13), (13,14), (14,15), (15,16),
                      (13,17), (0,17), (17,18), (18,19), (19,20)]
        left_hand_edges = [(s + 26, e + 26) for s, e in hand_links]
        right_hand_edges = [(s + 47, e + 47) for s, e in hand_links]
        connection_edges = [(15, 26), (16, 47)]
        
        return pose_edges + face_edges + left_hand_edges + right_hand_edges + connection_edges

    def _get_adjacency_matrix(self):
        A = np.zeros((self.num_node, self.num_node))
        for i, j in self.edges:
            A[i, j] = 1; A[j, i] = 1
        return torch.tensor(A, dtype=torch.float32)

class CleanBanglaDataset(Dataset):
    def __init__(self, split_dir, class_to_id):
        self.samples = []
        
        print(f"📂 Booting RAM-Cache Loader for {os.path.basename(split_dir)}...")
        for class_name in os.listdir(split_dir):
            class_path = os.path.join(split_dir, class_name)
            if not os.path.isdir(class_path): continue
            
            if class_name in class_to_id:
                cid = class_to_id[class_name]
                for f_name in os.listdir(class_path):
                    if f_name.endswith('.npy'):
                        f_path = os.path.join(class_path, f_name)
                        
                        raw_data = np.load(f_path).reshape(90, 68, 3) 
                        raw_data = raw_data - np.mean(raw_data, axis=1, keepdims=True) 
                        raw_data = raw_data.transpose(2, 0, 1)
                        
                        tensor_data = torch.tensor(raw_data, dtype=torch.float32)
                        tensor_label = torch.tensor(cid, dtype=torch.long)
                        self.samples.append((tensor_data, tensor_label))
        print(f"✅ Successfully cached {len(self.samples)} PyTorch Tensors in Memory!")
                        
    def __len__(self): return len(self.samples)

    def __getitem__(self, idx): return self.samples[idx]

class SpatialGraphConv(nn.Module):
    def __init__(self, in_c, out_c, A):
        super().__init__()
        self.register_buffer('A', A)
        self.conv = nn.Conv2d(in_c, out_c, 1)
    def forward(self, x):
        x = torch.einsum('nctv,vw->nctw', (x, self.A))
        return self.conv(x)

# 🟢 Channel Attention Module - Updated to GELU
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=4):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.GELU(),  # 🟢 GELU Activation
            nn.Linear(in_channels // reduction, in_channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class STGCN_Block(nn.Module):
    def __init__(self, in_c, out_c, A, stride=1, dropout=0.4):
        super().__init__()
        self.sgcn = SpatialGraphConv(in_c, out_c, A)
        self.tgcn = nn.Sequential(
            nn.BatchNorm2d(out_c), 
            nn.GELU(),  # 🟢 GELU Activation
            nn.Conv2d(out_c, out_c, (9, 1), (stride, 1), (4, 0)),
            nn.BatchNorm2d(out_c), 
            nn.Dropout(dropout)
        )
        self.attention = ChannelAttention(out_c)
        self.res = nn.Sequential(nn.Conv2d(in_c, out_c, 1, (stride, 1)), nn.BatchNorm2d(out_c)) if in_c != out_c or stride != 1 else nn.Identity()
        
    def forward(self, x): 
        sgcn_out = self.sgcn(x)
        tgcn_out = self.tgcn(sgcn_out)
        attended_out = self.attention(tgcn_out)
        # 🟢 Forward pass output updated to GELU
        return F.gelu(attended_out + self.res(x)) 

class BanglaSignSTGCN(nn.Module):
    def __init__(self, num_classes, A, dropout_rate=0.4):
        super().__init__()
        self.layer1 = STGCN_Block(3, 64, A, dropout=dropout_rate)
        self.layer2 = STGCN_Block(64, 64, A, dropout=dropout_rate)
        self.layer3 = STGCN_Block(64, 64, A, dropout=dropout_rate)
        self.layer4 = STGCN_Block(64, 128, A, stride=2, dropout=dropout_rate)
        self.layer5 = STGCN_Block(128, 128, A, dropout=dropout_rate)
        self.layer6 = STGCN_Block(128, 128, A, dropout=dropout_rate)
        self.layer7 = STGCN_Block(128, 256, A, stride=2, dropout=dropout_rate)
        self.layer8 = STGCN_Block(256, 256, A, dropout=dropout_rate)
        self.layer9 = STGCN_Block(256, 256, A, dropout=dropout_rate)
        self.fcn = nn.Conv2d(256, num_classes, 1)

    def forward(self, x):
        for l in [self.layer1,self.layer2,self.layer3,self.layer4,self.layer5,self.layer6,self.layer7,self.layer8,self.layer9]: x = l(x)
        x = F.avg_pool2d(x, x.size()[2:])
        return self.fcn(x).view(x.size(0), -1)

# ==========================================
# 2. TRAINING ENGINE
# ==========================================
def train_attention_model():
    seed = 42
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    DATASET_DIR = r"C:\Users\User\Documents\Personal Akams\Thesis\Final_Thesis_Dataset_1\Fold_1"
    TRAIN_DIR = os.path.join(DATASET_DIR, "train")
    VAL_DIR = os.path.join(DATASET_DIR, "val")
    
    MODELS_DIR = "Ablation_9Layer_SE_Gelu_Noise0.005_LS0.1"
    os.makedirs(MODELS_DIR, exist_ok=True)
    
    EPOCHS = 50
    BATCH_SIZE = 32
    LEARNING_RATE = 0.001
    DROPOUT = 0.4
    WEIGHT_DECAY = 1e-4
    NOISE_LEVEL = 0.005  
    
    target_idx = 0
    print("\n🔍 Scanning available DirectML GPUs...")
    for i in range(torch_directml.device_count()):
        gpu_name = torch_directml.device_name(i)
        print(f"  Found Device {i}: {gpu_name}")
        if "7900" in gpu_name or "GRE" in gpu_name or "RX" in gpu_name: 
            target_idx = i

    dml = torch_directml.device(target_idx)
    print(f"\n🚀 EXPERIMENT STARTED. Locked onto: {torch_directml.device_name(target_idx)}")
    print(f"🔬 Configuration: 9-Layer, SE Attention, GELU, Drop 0.4, Noise 0.005, LS 0.1")
    print(f"🔒 Random Seed locked to {seed} for strict reproducibility.")

    all_classes = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
    class_to_id = {cls_name: idx for idx, cls_name in enumerate(all_classes)}
    id_to_class = {idx: cls_name for cls_name, idx in class_to_id.items()} 
    num_classes = len(all_classes)
    
    homophone_pairs = [
        ("0_Lefthand", "O_Lefthand"), ("0_Righthand", "O_Righthand"),
        ("2_Lefthand", "V_Lefthand"), ("2_Righthand", "V_Righthand")
    ]
    
    allowed_confusions = []
    for a, b in homophone_pairs:
        if a in class_to_id and b in class_to_id:
            allowed_confusions.append(set([class_to_id[a], class_to_id[b]]))

    base_allowed_confusions = []
    for a, b in homophone_pairs:
        base_a = a.split('_')[0] if '_' in a else a
        base_b = b.split('_')[0] if '_' in b else b
        pair = set([base_a, base_b])
        if pair not in base_allowed_confusions:
            base_allowed_confusions.append(pair)

    train_data = CleanBanglaDataset(TRAIN_DIR, class_to_id)
    val_data = CleanBanglaDataset(VAL_DIR, class_to_id)

    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=False)
    val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

    graph = Graph()
    model = BanglaSignSTGCN(num_classes, graph.A, dropout_rate=DROPOUT).to(dml)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, foreach=False)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    
    # 🟢 Label Smoothing 0.1
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    best_val_acc = 0.0

    history_train = []
    history_strict = []
    history_relaxed = []
    history_base = []

    for epoch in range(EPOCHS):
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        
        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs = inputs.to(dml, non_blocking=True)
            labels = labels.to(dml, non_blocking=True)
            
            if NOISE_LEVEL > 0.0:
                noise = torch.randn(*inputs.shape, device=dml) * NOISE_LEVEL
                inputs = inputs + noise
                
            optimizer.zero_grad(set_to_none=True)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * inputs.size(0)
            _, pred = outputs.max(1)
            train_total += labels.size(0)
            train_correct += pred.eq(labels).sum().item()

            if epoch == 0 and (batch_idx + 1) % 100 == 0:
                print(f"   ⏳ Epoch [{epoch+1}/{EPOCHS}] - Processing Batch {batch_idx+1}/{len(train_loader)}...")

        epoch_train_acc = 100. * train_correct / train_total

        model.eval()
        val_loss, strict_correct, relaxed_correct, base_sign_correct, val_total = 0.0, 0, 0, 0, 0
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(dml, non_blocking=True)
                labels = labels.to(dml, non_blocking=True)
                
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                val_total += labels.size(0)
                
                _, top1_preds = outputs.max(1) 
                
                for i in range(labels.size(0)):
                    true_id = labels[i].item()
                    top1_id = top1_preds[i].item()
                    
                    true_name = id_to_class[true_id]
                    pred_name = id_to_class[top1_id]
                    
                    if top1_id == true_id:
                        strict_correct += 1
                        relaxed_correct += 1
                    else:
                        prediction_pair = set([true_id, top1_id])
                        if prediction_pair in allowed_confusions:
                            relaxed_correct += 1
                            
                    true_base = true_name.split('_')[0] if '_' in true_name else true_name
                    pred_base = pred_name.split('_')[0] if '_' in pred_name else pred_name
                    
                    if true_base == pred_base or set([true_base, pred_base]) in base_allowed_confusions:
                        base_sign_correct += 1
                
        strict_acc = 100. * strict_correct / val_total
        relaxed_acc = 100. * relaxed_correct / val_total
        base_sign_acc = 100. * base_sign_correct / val_total 

        history_train.append(epoch_train_acc)
        history_strict.append(strict_acc)
        history_relaxed.append(relaxed_acc)
        history_base.append(base_sign_acc)

        print(f"Epoch [{epoch+1:02d}/{EPOCHS}] | Train: {epoch_train_acc:.2f}% | "
              f"Val Strict: {strict_acc:.2f}% | Relaxed: {relaxed_acc:.2f}% -> Base Sign: {base_sign_acc:.2f}%")

        if relaxed_acc > best_val_acc: 
            best_val_acc = relaxed_acc
            save_path = os.path.join(MODELS_DIR, "9layer_SE_Gelu_Noise0.005_LS0.1.pth")
            torch.save(model.state_dict(), save_path)
            
        scheduler.step()

    print("\n" + "="*60)
    print(f"🎉 Training Complete! Best Relaxed Val Accuracy: {best_val_acc:.2f}%")
    print("="*60)

    print(f"⏳ Generating Detailed Accuracy Graph in {MODELS_DIR}...")
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, EPOCHS + 1), history_train, label='Train Accuracy', color='blue')
    plt.plot(range(1, EPOCHS + 1), history_strict, label='Simple Val Acc', color='red')
    plt.plot(range(1, EPOCHS + 1), history_relaxed, label='Val Relaxed', color='green')
    plt.plot(range(1, EPOCHS + 1), history_base, label='Val Base Sign', color='orange', linestyle='--')
    
    plt.title('9-Layer SE (GELU) Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True)
    
    graph_path_detailed = os.path.join(MODELS_DIR, "accuracy_graph_detailed.png")
    plt.savefig(graph_path_detailed)
    plt.close()
    print(f"✅ Saved Detailed Graph to {graph_path_detailed}")

    print(f"⏳ Generating Traditional Accuracy Graph in {MODELS_DIR}...")
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, EPOCHS + 1), history_train, label='Train Accuracy', color='blue')
    plt.plot(range(1, EPOCHS + 1), history_strict, label='Val Accuracy', color='red') 
    
    plt.title('9-Layer SE (GELU) Accuracy (Train vs Validation)')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True)
    
    graph_path_traditional = os.path.join(MODELS_DIR, "accuracy_graph_traditional.png")
    plt.savefig(graph_path_traditional)
    plt.close()
    print(f"✅ Saved Traditional Graph to {graph_path_traditional}")

if __name__ == "__main__":
    import multiprocessing
    multiprocessing.freeze_support()
    train_attention_model()


🔍 Scanning available DirectML GPUs...
  Found Device 0: AMD Radeon(TM) Graphics 
  Found Device 1: AMD Radeon RX 7900 GRE 

🚀 EXPERIMENT STARTED. Locked onto: AMD Radeon RX 7900 GRE 
🔬 Configuration: 9-Layer, SE Attention, GELU, Drop 0.4, Noise 0.005, LS 0.1
🔒 Random Seed locked to 42 for strict reproducibility.
📂 Booting RAM-Cache Loader for train...
✅ Successfully cached 34200 PyTorch Tensors in Memory!
📂 Booting RAM-Cache Loader for val...
✅ Successfully cached 8681 PyTorch Tensors in Memory!
   ⏳ Epoch [1/50] - Processing Batch 100/1069...
   ⏳ Epoch [1/50] - Processing Batch 200/1069...
   ⏳ Epoch [1/50] - Processing Batch 300/1069...
   ⏳ Epoch [1/50] - Processing Batch 400/1069...
   ⏳ Epoch [1/50] - Processing Batch 500/1069...
   ⏳ Epoch [1/50] - Processing Batch 600/1069...
   ⏳ Epoch [1/50] - Processing Batch 700/1069...
   ⏳ Epoch [1/50] - Processing Batch 800/1069...
   ⏳ Epoch [1/50] - Processing Batch 900/1069...
   ⏳ Epoch [1/50] - Processing Batch 1000/1069...
Epoch [